# COMP5329 — Week 12 Self-Study Material
## Deep Generative Models: A Complete Tour

A standalone, self-contained deep dive into every topic covered in the Week 12 lecture *Deep Generative Models* (A/Prof Chang Xu). This document is **not a supplement** to the Tutorial 12 notebook — it is intended to be read on its own, and is the primary reference for the lecture material.

> **Companion file.** `Week12_Deep_Generative_Models.ipynb` is the in-class tutorial; it contains a 60-min lesson plan, a coding exercise, and exam-style questions. This document is what you should read **before/after** that tutorial if you want to actually understand the material.


### How to use this document

1. **Read in order.** Each part builds on the previous — Parts 4 (DDPM), 5 (SDE), 6 (Flow), 7 (Consistency) form a single tightly linked story.
2. **Run the code.** Every model in this notebook is trained on the same toy dataset (8 Gaussians on a circle), so you can directly compare what each method does.
3. **Two layers per section.** Each section opens with the **lecture-level theory** (slides + derivations), then drops to the **engineering view** (the code that makes it work).
4. **Asides go beyond the slides** — you can skip them on first reading and come back when you need depth.

### Prerequisites
- Probability: joint, conditional, marginal distributions; KL divergence; expectations.
- Maximum likelihood and the difference between MLE and posterior inference.
- Backpropagation, autograd, MSE/BCE losses, batch normalisation.
- The Week 7 transformer / sinusoidal positional encoding material (we reuse it for diffusion timestep embeddings).

### Notation
| Symbol | Meaning |
|---|---|
| $x \in \mathbb{R}^d$ | a data point |
| $z$ | a latent variable (or the noise input to a generator/flow) |
| $p_{\text{data}}$ | the true (unknown) data distribution |
| $p_\theta$ | a model distribution with parameters $\theta$ |
| $q_\phi$ | an inference / encoder distribution with parameters $\phi$ |
| $\mathcal N(\mu, \Sigma)$ | Gaussian with mean $\mu$, covariance $\Sigma$ |
| $\nabla_x \log p(x)$ | the **score** of $p$ at $x$ |

---


## Part 0 — What Is a Generative Model?

A generative model for data $x \sim p_{\text{data}}$ does (at least) one of the following:

1. **Density estimation.** Given $x$, return $\hat p(x)$, an estimate of how likely $x$ is.
2. **Sample generation.** Produce fresh samples $\tilde x \sim \hat p$ that look like real data.

Slide 2 of the lecture splits these explicitly: density estimation gives you a *number* per input; sample generation gives you *new datapoints*. In modern deep learning, sample generation is what users usually want, but density estimation is how we *evaluate* and *invert* models.

### 0.1 Why we care (slide 3)

| Use case | Example |
|---|---|
| Realistic synthesis | Stable Diffusion, Imagen, Sora |
| Planning / world models | Simulating possible futures (stock-market, robotics) |
| Representation learning | Pre-training features that transfer to classification/segmentation |
| Inverse problems | Super-resolution, in-painting, MRI reconstruction (posterior sampling under a generative prior) |
| Scientific discovery | Molecule, protein, material design |

### 0.2 The two hard problems

Every generative model has to balance:
- **Tractable training** — you must be able to compute the loss from finite data, without intractable integrals or partition functions.
- **Tractable sampling** — once trained, generating $\tilde x$ has to be cheap and high-quality.

These are in tension. Almost every model in this document is a different compromise.

| Family | Has $p(x)$? | Easy sampling? | Parameterisation |
|---|---|---|---|
| Autoregressive (PixelCNN, GPT) | Exact | Slow (sequential) | $\prod_i p(x_i \mid x_{<i})$ |
| Normalising flows | Exact | Fast | Invertible $f_\theta$ |
| Energy-based models | Up to $Z$ | Hard (MCMC) | $e^{-E_\theta(x)}/Z$ |
| **VAE** (Part 3) | ELBO bound | Fast | encoder $q_\phi$, decoder $p_\theta$ |
| **GAN / WGAN** (Parts 1–2) | None | Fast | generator $G$, discriminator $D$ |
| **Diffusion / Score / Flow / Consistency** (Parts 4–7) | Implicit, recoverable via PF-ODE | Iterative → 1-step | score / velocity / consistency function |

This document focuses on the bottom three rows — they are the dominant families used today.

### 0.3 The 8-Gaussians toy

We use a simple 2-D toy throughout: 8 Gaussians arranged on a circle of radius 2. It is the smallest dataset that simultaneously stresses **mode coverage** (a model that collapses to one or two modes is visibly wrong) and **mode separation** (the gaps between modes have $p_{\text{data}} \approx 0$, which exposes failures of score-based methods at low density).


In [ ]:
# ── Imports and shared 2D dataset ───────────────────────────────────────────
import math
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline

# ── Shared 2D dataset: 8 Gaussians in a circle ───────────────────────────
def make_8gaussians(n_samples=10000, std=0.05):
    """Generate 2D data from 8 Gaussians arranged in a circle."""
    angles = torch.linspace(0, 2 * math.pi, 9)[:-1]  # 8 modes
    centres = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 2.0
    # Assign each sample to a random mode
    idx = torch.randint(0, 8, (n_samples,))
    data = centres[idx] + torch.randn(n_samples, 2) * std
    return data

torch.manual_seed(42)
data_2d = make_8gaussians(10000)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), s=1, alpha=0.3)
ax.set_title('Target Distribution: 8 Gaussians')
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

---
## Part 1 — Generative Adversarial Networks
*(Lecture slides 4–21)*

### 1.1 The two-player game (slides 6–11)

GANs sidestep density entirely. Instead of writing down $p_g(x)$, we just want **samples**. Two networks play a zero-sum game:

- **Generator** $G_\theta$ maps noise $z\sim p(z)$ (usually $\mathcal N(0,I)$) to a fake sample $G_\theta(z) \in \mathbb R^d$.
- **Discriminator** $D_\phi$ takes a sample $x$ and outputs $D_\phi(x) \in (0,1)$ — its predicted probability that $x$ is real.

The lecture analogy: **counterfeiter vs police**. The counterfeiter ($G$) wants to print fake currency that fools the police; the police ($D$) want to spot fakes. As both improve, the counterfeit becomes indistinguishable from real money.

> **Big idea.** We never write a formula for $p_g(x)$. We only need $G$ to be **differentiable**, so that we can backprop through it. The "model" is *implicit*.

### 1.2 The minimax objective (slide 12)

Goodfellow et al. (2014) wrote down

$$\boxed{\;\min_{\theta}\, \max_{\phi}\;\;V(D_\phi, G_\theta)\;=\;\mathbb{E}_{x\sim p_{\text{data}}}\!\left[\log D_\phi(x)\right]\;+\;\mathbb{E}_{z\sim p(z)}\!\left[\log\!\big(1 - D_\phi(G_\theta(z))\big)\right]\;}$$

Read it as:
- $D$ wants $D(x)\to 1$ on real data and $D(G(z))\to 0$ on fakes — this **maximises** $V$.
- $G$ wants $D(G(z))\to 1$ — this **minimises** the second term.

Training alternates between gradient ascent on $\phi$ (one or several steps) and gradient descent on $\theta$ (one step) — the algorithm on slide 13.

### 1.3 What the optimal $D$ looks like

Hold $G$ fixed and maximise pointwise. For each $x$ the integrand is

$$p_{\text{data}}(x)\log D(x) + p_g(x)\log(1 - D(x)).$$

Setting the derivative w.r.t. $D(x)$ to zero gives

$$\boxed{\;D^\star(x)\;=\;\frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}\;}$$

Plugging $D^\star$ back into $V$ and rearranging,

$$V(D^\star, G) \;=\; -\log 4 \;+\; 2\,\mathrm{JSD}\!\left(p_{\text{data}}\,\Vert\,p_g\right).$$

So when $D$ is optimal, training $G$ is **minimising the Jensen–Shannon divergence** between $p_{\text{data}}$ and $p_g$. The global optimum is $p_g = p_{\text{data}}$; at that point $D^\star \equiv 1/2$ and the discriminator is a coin flip — exactly the cartoon on slides 14–15.

> **Aside — JS vs KL.** $\mathrm{JSD}(P\|Q) = \tfrac12 \mathrm{KL}(P\|M) + \tfrac12 \mathrm{KL}(Q\|M)$ with $M=\tfrac12(P+Q)$. Unlike KL, JS is symmetric and always finite. But as we'll see in Part 2, on disjoint manifolds it is **constant**, which kills its gradient.

### 1.4 Why training is fragile (slides 23–24)

The "GAN minimises JS divergence" story is *only* true at $D = D^\star$. The empirical failure mode is the opposite:

- If $D$ is **too good**, $D(G(z)) \to 0$ everywhere, so $\log(1 - D(G(z))) \to \log 1 = 0$ and $\nabla_\theta \log(1-D(G(z))) \to 0$. The generator gets no signal.
- A common workaround is the **non-saturating loss** $\max_\theta \mathbb E_z[\log D(G(z))]$ — same fixed point but a steeper gradient when $D(G(z))$ is small.

The deeper problem (slide 26): when $p_{\text{data}}$ and $p_g$ live on (near-)disjoint low-dimensional manifolds — the typical case for natural images — $D$ can perfectly separate them with confidence 1, $\mathrm{JSD}$ is *constant* equal to $\log 2$, and the gradient is exactly zero. Constant loss ⇒ no learning. This was the key observation of Arjovsky et al. (Wasserstein GAN, 2017), motivating Part 2.

### 1.5 DCGAN: the architectural recipe (slides 20–21)

Radford et al. (2016) found a set of conventions that *empirically* stabilise image GANs. They are not theoretical, but they are why GANs that work on images became possible.

- Replace pooling with **strided convolutions** in $D$ and **fractional-strided / transposed convs** in $G$.
- Use **BatchNorm** in both $G$ and $D$ (except $G$'s output layer and $D$'s input layer).
- Remove fully-connected hidden layers in deeper architectures.
- **ReLU** in $G$ (Tanh on output); **LeakyReLU** in $D$.

These tricks have since been refined in StyleGAN, BigGAN, and so on, but the recipe still informs modern image generators.

### 1.6 Code: vanilla GAN on the 8-Gaussians toy

The implementation below trains a tiny GAN on the same 2-D dataset we'll reuse for every other model in this notebook. Watch how the discriminator and generator losses oscillate — that oscillation is GAN training in a nutshell.


In [ ]:
# ── GAN implementation ───────────────────────────────────────────────────────

class Generator(nn.Module):
    def __init__(self, latent_dim=2, data_dim=2, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, data_dim))
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1))
    def forward(self, x): return self.net(x)

In [ ]:
# ── GAN training ───────────────────────────────────────────────────────────
torch.manual_seed(42)
G = Generator(); D = Discriminator()
opt_G = torch.optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.999))
d_losses, g_losses = [], []

for step in range(5000):
    # ── Train Discriminator ──
    idx = torch.randperm(len(data_2d))[:256]
    real = data_2d[idx]
    z = torch.randn(256, 2)
    fake = G(z).detach()
    d_real = torch.sigmoid(D(real))
    d_fake = torch.sigmoid(D(fake))
    d_loss = -(torch.log(d_real + 1e-8).mean() + torch.log(1 - d_fake + 1e-8).mean())
    opt_D.zero_grad(); d_loss.backward(); opt_D.step()

    # ── Train Generator ──
    z = torch.randn(256, 2)
    fake = G(z)
    d_fake = torch.sigmoid(D(fake))
    g_loss = -torch.log(d_fake + 1e-8).mean()
    opt_G.zero_grad(); g_loss.backward(); opt_G.step()

    d_losses.append(d_loss.item()); g_losses.append(g_loss.item())
    if (step + 1) % 1000 == 0:
        print(f'  GAN step {step+1}, D loss: {d_loss.item():.3f}, G loss: {g_loss.item():.3f}')

In [ ]:
# ── GAN visualisation ────────────────────────────────────────────────────────
with torch.no_grad():
    gan_samples = G(torch.randn(2000, 2)).numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.3)
axes[0].set_title('Real Data')
axes[1].scatter(gan_samples[:, 0], gan_samples[:, 1], s=1, alpha=0.3, c='red')
axes[1].set_title('GAN Samples (sharper, but mode collapse?)')
axes[2].plot(d_losses[::10], alpha=0.6, label='D loss')
axes[2].plot(g_losses[::10], alpha=0.6, label='G loss')
axes[2].set_title('Training Dynamics (unstable!)'); axes[2].legend()
for ax in axes[:2]: ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

**Things to look at in the plot.**
- Are all 8 modes covered, or did the generator collapse to a few?
- Look at $D$ and $G$ losses over time. Healthy GAN training looks like a noisy oscillation; if either loss flatlines, you are stuck.

> **Mode collapse** is when $G$ learns to map every $z$ to a single mode of $p_{\text{data}}$. The discriminator can still be fooled (because that mode is real), so the loss doesn't blow up — but coverage is destroyed. There's no objective term that explicitly punishes this in the vanilla GAN, which is one of the reasons people invented Wasserstein GAN, unrolled GAN, mini-batch discrimination, packed GAN, etc.


---
## Part 2 — Wasserstein GAN
*(Lecture slides 22–35)*

### 2.1 Why WGAN exists: the support-overlap problem

Recall from §1.4: when $p_{\text{data}}$ and $p_g$ are supported on disjoint low-dimensional manifolds, JS divergence is **constant** equal to $\log 2$. Slide 26 makes this precise:

> *If the supports of $p_{\text{data}}$ and $p_g$ have no overlap, then $\mathrm{JSD}(p_{\text{data}}\|p_g) = \log 2$, which has zero gradient w.r.t. the generator parameters.*

This is the typical case in high dimensions (a manifold-vs-manifold intersection generically has measure zero), and it explains why GANs train unstably even when they "should" work.

### 2.2 A better distance: Earth-Mover (Wasserstein-1) (slides 27–29)

Wasserstein-1, also called the **Earth-Mover (EM) distance**, is

$$W_1(P, Q) \;=\; \inf_{\gamma \in \Pi(P, Q)} \mathbb E_{(x,y)\sim \gamma}\big[\|x - y\|\big],$$

where $\Pi(P,Q)$ is the set of all *couplings* — joint distributions with marginals $P$ and $Q$. Intuitively, $\gamma$ is a *transport plan* that says how much "mass" to move from $x$ to $y$, and $\|x-y\|$ is the cost. $W_1$ is the cheapest plan.

The crucial property (slide 28): if $P$ and $Q$ are two delta distributions at distance $d$, $W_1(P,Q) = d$, which **smoothly varies as you move them**, even when their supports are disjoint. Compare with JS, which is $\log 2$ as soon as they don't overlap and **drops discontinuously to 0** when they coincide. JS has no gradient; $W_1$ does.

### 2.3 Kantorovich–Rubinstein duality (slide 29)

Computing the infimum over couplings is intractable in high dimensions. The duality theorem gives an equivalent **supremum** form:

$$W_1(P, Q) \;=\; \sup_{\|f\|_L \le 1}\;\mathbb E_{x\sim P}[f(x)] - \mathbb E_{y\sim Q}[f(y)],$$

where the sup is over all **1-Lipschitz** functions $f:\mathbb R^d \to \mathbb R$ (functions whose gradient norm is at most 1 everywhere). For any $K \ge 1$, restricting to $K$-Lipschitz functions multiplies the answer by $K$, so we can work with $K$-Lipschitz functions and rescale — only the **Lipschitz constraint** matters.

The dual is great because it is a **maximisation over functions** — exactly what a neural network can parameterise.

### 2.4 The WGAN objective (slides 30–31)

Let $f_w$ be a network with weights $w$ that is constrained to be $K$-Lipschitz (we will see how shortly). Then $W_1(p_{\text{data}}, p_g)$ is approximated by

$$\hat W_1 \;=\; \max_{w \in \mathcal W}\;\mathbb E_{x\sim p_{\text{data}}}[f_w(x)] - \mathbb E_{z\sim p(z)}[f_w(G_\theta(z))].$$

Note that **there is no log and no sigmoid**. The network $f_w$ is called the **critic** (not discriminator) because it doesn't classify — it estimates how much one distribution is *worth* relative to another, in EM units.

The training objective is then

$$\min_\theta\; \max_{w\in\mathcal W}\;\mathbb E_{x\sim p_{\text{data}}}[f_w(x)] - \mathbb E_{z}[f_w(G_\theta(z))].$$

### 2.5 Enforcing the Lipschitz constraint: weight clipping (slide 30)

The original WGAN paper enforces $f_w$ Lipschitz the bluntest way possible: **clip every weight to a box** $[-c, c]$ after each critic update. This gives $\mathcal W = [-c, c]^{|w|}$, which is a compact set; any function on a compact set parameterised by bounded weights is Lipschitz, and the constant is bounded by some product of layer widths and clip values.

Weight clipping is crude. WGAN-GP (Gulrajani et al., 2017) replaced it with a **gradient penalty** — adding $\lambda(\|\nabla_x f_w(x)\| - 1)^2$ to the critic loss at points along straight lines between real and fake samples. We'll keep clipping in the code below to stay faithful to the lecture slides.

### 2.6 The WGAN algorithm (slide 32)

```
for each iteration:
    for n_critic steps:
        sample real batch x
        sample noise batch z
        loss_C = -(mean(f_w(x)) - mean(f_w(G(z))))    # negate to minimise
        opt_C.step(); clip every weight to [-c, c]
    sample noise batch z
    loss_G = -mean(f_w(G(z)))
    opt_G.step()
```

Two important hyperparameters:
- `n_critic = 5`: train $f_w$ harder than $G$, because we want the dual to be tight before each generator step.
- Use **RMSProp / SGD without momentum** — the original paper found Adam unstable with weight clipping (the clipping interacts badly with momentum's running averages).

### 2.7 Why WGAN's loss is *meaningful* (slides 33–35)

A killer property: because $\hat W_1$ is an *estimate of a real metric*, the critic loss **correlates with sample quality**. Slides 34–35 show that as $f_w$'s estimate of $W_1(p_{\text{data}}, p_g)$ decreases over training, samples visibly get better. Vanilla GAN losses do *not* track quality at all (slide 35 shows the JS estimate going **up** while DCGAN samples are getting better) — the loss is uninformative as a debugging signal.

This is why most modern image generators (StyleGAN, BigGAN, even some diffusion baselines) use Wasserstein-style losses with gradient penalties.

### 2.8 Code: minimal WGAN-clip on the 8-Gaussians toy

This is a simple WGAN with weight clipping. Compare with the vanilla GAN above — WGAN is slower (5 critic steps per generator step) but more stable.


In [ ]:
# ── WGAN-clip implementation ─────────────────────────────────────────────────

class Critic(nn.Module):
    """WGAN critic — note: no sigmoid on the output."""
    def __init__(self, data_dim=2, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden,   hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1))         # scalar score, NOT a probability
    def forward(self, x): return self.net(x)


# ── WGAN training loop ──────────────────────────────────────────────────────
torch.manual_seed(7)
G_w = Generator()           # reuse the Generator class from the GAN cell above
crit = Critic()
opt_G = torch.optim.RMSprop(G_w.parameters(),  lr=5e-5)
opt_C = torch.optim.RMSprop(crit.parameters(), lr=5e-5)

n_critic = 5     # train critic 5x more often than generator
clip     = 0.01  # weight-clip range

w_dist_history, gen_loss_history = [], []
for step in range(2000):
    # ---- critic updates ----
    for _ in range(n_critic):
        idx  = torch.randperm(len(data_2d))[:256]
        real = data_2d[idx]
        fake = G_w(torch.randn(256, 2)).detach()
        loss_C = -(crit(real).mean() - crit(fake).mean())   # maximise W ⇒ minimise -W
        opt_C.zero_grad(); loss_C.backward(); opt_C.step()
        for p in crit.parameters():
            p.data.clamp_(-clip, clip)

    # ---- generator update ----
    z = torch.randn(256, 2)
    loss_G = -crit(G_w(z)).mean()
    opt_G.zero_grad(); loss_G.backward(); opt_G.step()

    if (step + 1) % 100 == 0:
        with torch.no_grad():
            w_est = (crit(data_2d[:512]).mean()
                     - crit(G_w(torch.randn(512, 2))).mean()).item()
        w_dist_history.append(w_est)
        gen_loss_history.append(-loss_G.item())
        if (step + 1) % 500 == 0:
            print(f'  WGAN step {step+1}: estimated W ≈ {w_est:.4f}')


# ── WGAN visualisation ─────────────────────────────────────────────────────
with torch.no_grad():
    wgan_samples = G_w(torch.randn(2000, 2)).numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.3)
axes[0].set_title('Real Data');                axes[0].set_aspect('equal')
axes[1].scatter(wgan_samples[:, 0], wgan_samples[:, 1], s=1, alpha=0.3, c='orange')
axes[1].set_title('WGAN Samples');             axes[1].set_aspect('equal')
axes[2].plot(w_dist_history, label='estimated $W_1$')
axes[2].set_xlabel('×100 critic updates'); axes[2].set_ylabel(r'$\hat W_1$')
axes[2].set_title('Critic estimate of $W_1$ — should decrease')
axes[2].legend()
plt.tight_layout(); plt.show()


**What to look for.**
- The WGAN samples should cover all 8 modes more reliably than the vanilla GAN — losing one or two is the typical vanilla-GAN failure.
- The "estimated $W_1$" curve should **decrease** over training. This is the magic of WGAN: the loss number itself is informative, not just a tug-of-war.
- Compare with the vanilla GAN losses two cells up — those oscillate without telling you much about quality.


---
## Part 3 — Variational Autoencoder
*(Lecture slides 36–46)*

### 3.1 Recap: the classical autoencoder (slides 37–40)

A classical autoencoder is two networks chained back-to-back:

$$\text{encoder}\;e_\phi:\;\mathbb R^d \to \mathbb R^k,\qquad \text{decoder}\;d_\theta:\;\mathbb R^k \to \mathbb R^d,$$

trained to minimise the reconstruction loss

$$\mathcal L_{\text{AE}}(\theta,\phi) \;=\; \mathbb E_{x\sim p_{\text{data}}}\!\left[\|x - d_\theta(e_\phi(x))\|_2^2\right].$$

With $k < d$ this is a non-linear PCA — it learns features that capture variation in the data. It's useful for representation learning (slide 39: throw away the decoder, keep the encoder, fine-tune for a downstream task).

**Why we cannot generate from a vanilla AE** (slide 40):
- The encoder maps each training point to a single code $z = e_\phi(x)$. There is no constraint on what the *distribution* of those codes looks like — they could form arbitrary clusters with empty space between them.
- If you sample $z$ from anywhere except those exact training codes, the decoder produces nonsense, because nothing during training told it what to do for unseen $z$.

We need a model where (a) the latent codes are forced to fill out a known distribution, so we can sample from it, and (b) the decoder is trained on samples drawn from that distribution. That model is the VAE.

### 3.2 The latent-variable setup (slide 41)

Assume the data comes from a latent variable model:

$$z \sim p(z),\qquad x \mid z \sim p_\theta(x \mid z),\qquad p_\theta(x) = \int p_\theta(x\mid z)\,p(z)\,dz.$$

Choices we make:
- $p(z) = \mathcal N(0, I)$ — a fixed standard Gaussian prior.
- $p_\theta(x\mid z)$ — Gaussian with mean given by a neural network $d_\theta(z)$, fixed variance (this is what gives the MSE reconstruction term in the loss).

The marginal $p_\theta(x)$ is what we want to maximise — but the integral is **intractable** for any non-trivial decoder. We need a way around it.

### 3.3 Deriving the ELBO

Introduce an **approximate posterior** $q_\phi(z\mid x)$ (the encoder, parameterised as a diagonal Gaussian):

$$q_\phi(z\mid x) \;=\; \mathcal N\!\big(\mu_\phi(x),\; \mathrm{diag}\,\sigma_\phi^2(x)\big).$$

Then for any $q$,

$$\log p_\theta(x) \;=\; \log \int p_\theta(x\mid z)\,p(z)\,dz \;=\; \log \mathbb E_{q(z\mid x)}\!\left[\frac{p_\theta(x\mid z)\,p(z)}{q(z\mid x)}\right].$$

Apply Jensen's inequality ($\log$ is concave, so $\log \mathbb E[X] \ge \mathbb E[\log X]$):

$$\log p_\theta(x) \;\ge\; \mathbb E_{q_\phi(z\mid x)}\!\left[\log p_\theta(x\mid z)\right] \;-\; \mathrm{KL}\!\left(q_\phi(z\mid x)\,\Vert\,p(z)\right).$$

This is the **Evidence Lower BOund (ELBO)**. Decomposed:

| Term | Interpretation |
|---|---|
| $\mathbb E_{q_\phi}[\log p_\theta(x\mid z)]$ | **Reconstruction**: the decoder must explain $x$ from $z$. |
| $-\mathrm{KL}(q_\phi(z\mid x)\,\Vert\,p(z))$ | **Regulariser**: the encoder must keep posteriors close to the prior. |

Maximising the ELBO simultaneously fits the decoder and forces the encoded codes to fill the prior — exactly what we needed.

> **Aside — what's the gap?** $\log p_\theta(x) - \mathrm{ELBO} = \mathrm{KL}(q_\phi(z\mid x)\,\Vert\,p_\theta(z\mid x))$. The ELBO is tight when the encoder matches the *true* posterior. In a perfect world we'd compute that posterior; in practice we make do with a Gaussian $q_\phi$.

### 3.4 The closed-form KL

For $q = \mathcal N(\mu, \mathrm{diag}\,\sigma^2)$ and $p = \mathcal N(0, I)$ in $k$ dimensions,

$$\mathrm{KL}(q\,\Vert\,p) \;=\; \tfrac12 \sum_{j=1}^{k}\!\left(\sigma_j^2 \;+\; \mu_j^2 \;-\; 1 \;-\; \log \sigma_j^2\right).$$

In code, since the encoder outputs `logvar = log σ²` for numerical stability,

```python
kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
```

This is the formula in `vae_loss` below. **Memorise it** — it is on every diffusion/VAE/flow exam.

### 3.5 The reparameterisation trick (slide 42)

The ELBO contains $\mathbb E_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)]$. A naive Monte Carlo estimate is

$$\frac1L \sum_{\ell=1}^L \log p_\theta(x \mid z^{(\ell)}),\quad z^{(\ell)} \sim q_\phi(z\mid x).$$

But **we cannot backprop through that sampling step.** `z = torch.normal(mu, sigma)` produces a tensor that has *no gradient connection* to `mu` and `sigma` — autograd doesn't know how to differentiate "draw a sample". The chain rule fails at the random node.

The **reparameterisation trick** rewrites the sampling so the randomness is in a *parameter-free* node:

$$z \;=\; \mu_\phi(x) \;+\; \sigma_\phi(x) \odot \varepsilon,\qquad \varepsilon \sim \mathcal N(0, I).$$

Now $\varepsilon$ has no parameters; it's a frozen input. $z$ is a deterministic differentiable function of $\mu_\phi$ and $\sigma_\phi$, so $\nabla_\phi$ flows through $z$ directly. The expectation becomes

$$\mathbb E_{q_\phi(z\mid x)}\!\left[\log p_\theta(x\mid z)\right] \;=\; \mathbb E_{\varepsilon\sim\mathcal N(0,I)}\!\left[\log p_\theta\!\big(x \mid \mu_\phi(x) + \sigma_\phi(x)\odot\varepsilon\big)\right],$$

which is differentiable in $\phi$ and gives an unbiased low-variance estimator with $L=1$ Monte Carlo sample per data point.

> **Why this is non-trivial.** Without the trick, you'd need a high-variance score-function estimator (REINFORCE): $\nabla_\phi \mathbb E_{q_\phi}[f(z)] = \mathbb E_{q_\phi}[f(z)\,\nabla_\phi \log q_\phi(z)]$. This is unbiased but its variance is so large that VAEs are essentially impossible to train with it. The reparameterisation trick is what made VAEs practical.

### 3.6 Training objective

Combining everything, the per-example VAE loss (negated ELBO) is

$$\mathcal L_{\text{VAE}}(x; \theta, \phi) \;=\; \underbrace{\|x - d_\theta(z)\|_2^2}_{\text{reconstruction}}\;+\;\underbrace{\tfrac12 \sum_{j}(\sigma_{\phi,j}^2 + \mu_{\phi,j}^2 - 1 - \log\sigma_{\phi,j}^2)}_{\text{KL to prior}},\quad z = \mu_\phi(x) + \sigma_\phi(x)\odot\varepsilon.$$

(The MSE term is what you get when $p_\theta(x\mid z) = \mathcal N(d_\theta(z), I)$ and you drop the constant $\log 2\pi$.)

### 3.7 Generation

After training, generation is a one-step process:
1. Sample $z \sim \mathcal N(0, I)$.
2. Return $\hat x = d_\theta(z)$.

That's it — VAEs are **single-step generators**. No iterative refinement. (We will spend Parts 4–7 explaining why and how diffusion models give up that single-step speed in exchange for sharper samples, then progressively recover it.)

### 3.8 Code: VAE on the 8-Gaussians toy


In [ ]:
# ── VAE implementation ───────────────────────────────────────────────────────

class VAE(nn.Module):
    def __init__(self, data_dim=2, latent_dim=2, hidden_dim=128):
        super().__init__()
        # Encoder: data -> (mu, logvar)
        self.enc = nn.Sequential(nn.Linear(data_dim, hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        # Decoder: z -> data
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim, data_dim))

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(x_recon, x, mu, logvar):
    """Negative ELBO = reconstruction + KL."""
    recon = F.mse_loss(x_recon, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + kl

In [ ]:
# ── VAE training ───────────────────────────────────────────────────────────
torch.manual_seed(42)
vae = VAE()
opt_vae = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(300):
    idx = torch.randperm(len(data_2d))[:256]
    x_batch = data_2d[idx]
    x_recon, mu, logvar = vae(x_batch)
    loss = vae_loss(x_recon, x_batch, mu, logvar)
    opt_vae.zero_grad(); loss.backward(); opt_vae.step()
    if (epoch + 1) % 100 == 0:
        print(f'  VAE epoch {epoch+1}, loss: {loss.item():.2f}')

In [ ]:
# ── VAE visualisation ────────────────────────────────────────────────────────
with torch.no_grad():
    z_samples = torch.randn(2000, 2)
    vae_samples = vae.decode(z_samples).numpy()
    mu_all, _ = vae.encode(data_2d[:2000])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.3)
axes[0].set_title('Real Data'); axes[0].set_xlim(-3.5, 3.5); axes[0].set_ylim(-3.5, 3.5)
axes[1].scatter(vae_samples[:, 0], vae_samples[:, 1], s=1, alpha=0.3, c='orange')
axes[1].set_title('VAE Samples (blurry!)'); axes[1].set_xlim(-3.5, 3.5); axes[1].set_ylim(-3.5, 3.5)
axes[2].scatter(mu_all[:, 0].numpy(), mu_all[:, 1].numpy(), s=1, alpha=0.3, c='green')
axes[2].set_title('Latent Space')
for ax in axes: ax.set_aspect('equal')
plt.tight_layout(); plt.show()
print('Note: VAE samples fill the gaps between modes -- characteristic blurriness.')

**What to look for.**
- The **right plot** is the encoded latent space. The 8 modes should encode to 8 visibly separated clusters in a 2-D ball — the KL term has pushed them inside the unit Gaussian.
- The **middle plot** is samples decoded from $\mathcal N(0,I)$. They should land near the 8 modes, but you may notice "interpolation tails" between modes — VAEs have this characteristic blurriness because the reconstruction loss is symmetric and averages over modes when $z$ is ambiguous.

### 3.9 VAE + GAN hybrids (slide 46)

A practical observation: VAE samples are **structured but blurry**, GAN samples are **sharp but unstructured**. People have tried to combine them:

- **VAE-GAN** (Larsen et al. 2016): replace the pixel-wise reconstruction loss with a feature-matching loss in the layers of a discriminator, so the decoder is judged on perceptual similarity, not pixel MSE.
- **Adversarial Autoencoder** (Makhzani et al. 2015): replace the KL term with an adversarial loss that forces $q_\phi(z)$ to match $p(z)$ — the discriminator runs on *latents*, not pixels.
- **VQ-VAE-2 + autoregressive prior**: encode to discrete codes with a VAE, then learn a powerful autoregressive prior on the codes. This is what underlies DALL-E v1 and many video generators.

The lecture mentions VAE+GAN in passing on slide 46; we don't go deeper here, but it's worth knowing the family exists.


---
## Part 4 — Denoising Diffusion Probabilistic Models (DDPM)
*(Lecture slides 47–70)*

### 4.1 Why diffusion: the showcase (slides 48–54)

Before getting into the maths, it is worth seeing *why* diffusion took over generative modeling in 2020–22:

| Application | Reference |
|---|---|
| Unconditional image synthesis (beats GANs) | Dhariwal & Nichol, "Diffusion Models Beat GANs on Image Synthesis", NeurIPS 2021 |
| Super-resolution | Saharia et al., "Image Super-Resolution via Iterative Refinement", T-PAMI 2023 |
| Image-to-image translation | Saharia et al., "Palette", NeurIPS 2021 |
| Text-to-image | Ramesh et al., "DALL·E 2", 2022; Saharia et al., "Imagen", 2022 |
| Text-to-animation | Disco Diffusion, 2022 |
| Text-to-video | Ho et al., "Video Diffusion Models", CVPR 2022 |

Every one of these is a denoising diffusion model with some conditioning bolted on. The core algorithm is what we develop now.

### 4.2 The two processes (slide 56)

A DDPM consists of **two Markov processes** that run in opposite directions in time:

- **Forward process** (fixed, no learning): take a clean datapoint $x_0$ and gradually add Gaussian noise over $T$ steps until $x_T$ is approximately pure Gaussian noise.
- **Reverse process** (learned): start from pure noise $x_T \sim \mathcal N(0, I)$ and gradually denoise to recover a clean sample $x_0$.

Conceptually:

$$\underbrace{x_0}_{\text{data}} \;\to\; x_1 \;\to\; \cdots \;\to\; x_T \;\approx\; \mathcal N(0, I) \;\to\; \cdots \;\to\; x_0\;\;\text{(reconstruction)}$$

### 4.3 The forward process (slides 57–58)

Each forward step adds a tiny bit of Gaussian noise:

$$q(x_t \mid x_{t-1}) \;=\; \mathcal N\!\left(x_t;\;\sqrt{1 - \beta_t}\,x_{t-1},\;\beta_t I\right),$$

where $\{\beta_t\}_{t=1}^T$ is the **noise schedule** — a sequence of small positive numbers (e.g. linearly increasing from $10^{-4}$ to $0.02$). The full forward chain is

$$q(x_{1:T} \mid x_0) \;=\; \prod_{t=1}^{T} q(x_t \mid x_{t-1}).$$

The mean is $\sqrt{1-\beta_t}\,x_{t-1}$ (slightly shrunken) and the variance is $\beta_t$ (a small additive noise). Why the $\sqrt{1-\beta_t}$ factor? It keeps the **variance constant** — if $\mathrm{Var}(x_{t-1}) = 1$ then

$$\mathrm{Var}(x_t) \;=\; (1 - \beta_t)\cdot 1 \;+\; \beta_t \;=\; 1.$$

So the data is "scaled and noised" rather than allowed to explode.

### 4.4 The diffusion kernel: closed-form noising (slides 59–60)

The killer property of this Markov chain: you can sample $x_t$ from $x_0$ **in one step**, without iterating. Define

$$\alpha_t \;=\; 1 - \beta_t,\qquad \bar\alpha_t \;=\; \prod_{s=1}^{t} \alpha_s.$$

Then

$$\boxed{\;q(x_t \mid x_0) \;=\; \mathcal N\!\left(x_t;\;\sqrt{\bar\alpha_t}\,x_0,\;(1 - \bar\alpha_t)\,I\right)\;}$$

equivalently $\;x_t \;=\; \sqrt{\bar\alpha_t}\,x_0 \;+\; \sqrt{1 - \bar\alpha_t}\,\varepsilon,\quad \varepsilon \sim \mathcal N(0,I).$

> **Proof (slide 60).** By induction. $x_1 = \sqrt{\alpha_1}\,x_0 + \sqrt{\beta_1}\,\varepsilon_1$. Plug into $x_2 = \sqrt{\alpha_2}\,x_1 + \sqrt{\beta_2}\,\varepsilon_2$:
> $$x_2 = \sqrt{\alpha_2}\sqrt{\alpha_1}\,x_0 + \sqrt{\alpha_2}\sqrt{\beta_1}\,\varepsilon_1 + \sqrt{\beta_2}\,\varepsilon_2.$$
> The two noise terms are independent Gaussians with combined variance $\alpha_2\beta_1 + \beta_2 = 1 - \alpha_2\alpha_1 = 1 - \bar\alpha_2$. So $x_2 = \sqrt{\bar\alpha_2}\,x_0 + \sqrt{1-\bar\alpha_2}\,\varepsilon$. The induction step is identical.

This single equation is what makes diffusion training cheap: instead of running $t$ forward steps to compute the loss at step $t$, we sample $t$ uniformly and noise once.

### 4.5 Noise schedules: linear vs cosine (slide 61)

Two common choices:
- **Linear** (Ho et al. 2020): $\beta_t$ linearly from $10^{-4}$ to $0.02$, $T = 1000$. Simple but $\bar\alpha_t$ collapses to 0 too quickly — most of the training signal lives in the first 1/4 of the timesteps and the rest is wasted on near-pure noise.
- **Cosine** (Nichol & Dhariwal, 2021): set $\bar\alpha_t = \cos^2\!\left(\frac{t/T + s}{1 + s}\cdot\frac\pi2\right)$ for a small offset $s$. The signal-to-noise ratio decays *linearly in $t$*, which spreads training signal evenly. This gave a noticeable FID improvement on ImageNet.

In our toy notebook we use a small linear schedule with $T = 300$, which is enough for 2-D data.


In [ ]:
# ── DDPM noise schedule ──────────────────────────────────────────────────────

T_diffusion = 300  # fewer steps for 2D (not 1000 -- sufficient for toy data)

def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

betas = linear_beta_schedule(T_diffusion)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# Plot alpha_bar decay
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(alpha_bars.numpy())
ax.set_xlabel('Timestep t'); ax.set_ylabel(r'$\bar{\alpha}_t$')
ax.set_title(r'Noise schedule: $\bar{\alpha}_t$ decays from 1 (clean) to ~0 (noise)')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── Forward process visualisation ───────────────────────────────────────────
x0 = data_2d[:500]
timesteps = [0, 30, 75, 150, 225, T_diffusion - 1]

fig, axes = plt.subplots(1, len(timesteps), figsize=(18, 3))
for i, t in enumerate(timesteps):
    ab = alpha_bars[t]
    xt = math.sqrt(ab) * x0 + math.sqrt(1 - ab) * torch.randn_like(x0)
    axes[i].scatter(xt[:, 0].numpy(), xt[:, 1].numpy(), s=1, alpha=0.3)
    axes[i].set_title(f't = {t}\n' + r'$\bar{\alpha}$' + f' = {ab:.3f}')
    axes[i].set_xlim(-4, 4); axes[i].set_ylim(-4, 4); axes[i].set_aspect('equal')
plt.suptitle('Forward Process: Data dissolves into noise', fontsize=13)
plt.tight_layout(); plt.show()

**Read the plots.** $\bar\alpha_t$ decays from 1 (signal preserved) to ~0 (signal gone). The forward-process scatter shows the data going from 8 sharp clusters to a single isotropic blob — that is exactly what we want, because then $\mathcal N(0, I)$ is a faithful starting point for the reverse process.

### 4.6 The reverse process (slide 62)

Reverse-time generation is

$$p_\theta(x_{0:T}) \;=\; p(x_T) \prod_{t=1}^{T} p_\theta(x_{t-1} \mid x_t),\qquad p(x_T) = \mathcal N(0, I).$$

In principle we'd like to know the *true* reverse $q(x_{t-1}\mid x_t)$. We can't — it depends on the unknown $p_{\text{data}}$. **However**, if we condition on $x_0$ as well, the posterior $q(x_{t-1}\mid x_t, x_0)$ is **tractable** and Gaussian. We then learn a model $p_\theta(x_{t-1}\mid x_t)$ to match it.

> **Why is the conditional reversible at all?** Slide 62 makes a key point: in the limit $\beta_t \to 0$, the reverse step is also Gaussian. Each noising step is so small that "going backwards" is also locally Gaussian. So we can parameterise $p_\theta$ as a Gaussian with learnable mean and (often fixed) variance.

### 4.7 The true reverse posterior: full derivation (slides 63–64)

Apply Bayes:

$$q(x_{t-1} \mid x_t, x_0) \;=\; \frac{q(x_t \mid x_{t-1})\,q(x_{t-1}\mid x_0)}{q(x_t\mid x_0)}.$$

All three factors on the right are known Gaussians:
- $q(x_t\mid x_{t-1}) = \mathcal N(\sqrt{\alpha_t}\,x_{t-1},\,\beta_t I)$
- $q(x_{t-1}\mid x_0) = \mathcal N(\sqrt{\bar\alpha_{t-1}}\,x_0,\,(1-\bar\alpha_{t-1})I)$
- $q(x_t\mid x_0) = \mathcal N(\sqrt{\bar\alpha_t}\,x_0,\,(1-\bar\alpha_t)I)$

Multiplying Gaussians and completing the square (a tedious but mechanical exercise — work through it once on paper) gives

$$q(x_{t-1}\mid x_t, x_0) \;=\; \mathcal N\!\big(\tilde\mu_t(x_t, x_0),\,\tilde\beta_t I\big),$$

where

$$\tilde\mu_t(x_t, x_0) \;=\; \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1 - \bar\alpha_t}\,x_0 \;+\; \frac{\sqrt{\alpha_t}(1 - \bar\alpha_{t-1})}{1 - \bar\alpha_t}\,x_t,$$

$$\tilde\beta_t \;=\; \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\,\beta_t.$$

So the *true* posterior mean is a convex combination of $x_0$ and $x_t$, with weights determined by the noise schedule.

### 4.8 The ELBO for diffusion (slide 65)

As with VAEs, we maximise a variational bound on $\log p_\theta(x_0)$. Sohl-Dickstein et al. (2015) and Ho et al. (2020) show that the ELBO simplifies to

$$\mathcal L = \mathbb E_q\!\left[\underbrace{\mathrm{KL}(q(x_T\mid x_0)\,\Vert\,p(x_T))}_{L_T,\;\text{constant}} + \sum_{t=2}^{T}\underbrace{\mathrm{KL}(q(x_{t-1}\mid x_t,x_0)\,\Vert\,p_\theta(x_{t-1}\mid x_t))}_{L_{t-1}} - \underbrace{\log p_\theta(x_0\mid x_1)}_{L_0}\right].$$

- $L_T$ is constant (no learnable parameters), drop it.
- Each $L_{t-1}$ is a KL between **two Gaussians** with the same variance, which simplifies to a scaled $\ell_2$ between their means.
- $L_0$ is a single-step decoder term (small, often folded into the discrete-pixel decoder).

Parameterising $p_\theta(x_{t-1}\mid x_t) = \mathcal N(\mu_\theta(x_t, t),\, \sigma_t^2 I)$, we find that $L_{t-1}$ is

$$L_{t-1} \;=\; \mathbb E_q\!\left[\frac{1}{2\sigma_t^2}\|\tilde\mu_t(x_t, x_0) - \mu_\theta(x_t, t)\|^2\right] + \text{const}.$$

So training is "make the model's mean match the *true* posterior mean".

### 4.9 The ε-prediction trick (slide 66)

Direct mean prediction is awkward. There's a much nicer parameterisation. Recall

$$x_t \;=\; \sqrt{\bar\alpha_t}\,x_0 \;+\; \sqrt{1-\bar\alpha_t}\,\varepsilon,$$

so $x_0 = \frac{1}{\sqrt{\bar\alpha_t}}\!\left(x_t - \sqrt{1-\bar\alpha_t}\,\varepsilon\right)$. Substitute this into $\tilde\mu_t(x_t, x_0)$ and after algebra:

$$\tilde\mu_t \;=\; \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon\right).$$

So if we parameterise

$$\mu_\theta(x_t, t) \;=\; \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t, t)\right),$$

then matching $\mu_\theta$ to $\tilde\mu_t$ is **equivalent to matching $\varepsilon_\theta$ to $\varepsilon$**. The ELBO term $L_{t-1}$ becomes

$$L_{t-1} \;\propto\; \mathbb E_{x_0,\varepsilon,t}\!\left[\|\varepsilon_\theta(x_t, t) - \varepsilon\|^2\right].$$

> **Why is ε-prediction better than $x_0$- or $\mu$-prediction?**
> - The target $\varepsilon$ is always a unit Gaussian regardless of $t$, so the loss has the same scale across timesteps.
> - At small $t$, $x_0 \approx x_t$, so $x_0$-prediction would have a near-zero target everywhere and provide little signal.
> - At large $t$, $x_0$ is hard to predict (almost no signal in $x_t$), while $\varepsilon$ — which is what dominates $x_t$ — is exactly what the network sees in its input. The loss landscape is much friendlier.

### 4.10 The simplified objective (slide 67)

Empirically, Ho et al. (2020) found that **dropping the per-timestep weighting** in the ELBO actually *improves* sample quality:

$$\boxed{\;\mathcal L_{\text{simple}}(\theta) \;=\; \mathbb E_{t,x_0,\varepsilon}\!\left[\|\varepsilon - \varepsilon_\theta(\sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon,\;t)\|^2\right]\;}$$

with $t \sim \mathrm{Uniform}\{1,\dots,T\}$. This is the loss that essentially every diffusion model paper since 2020 uses. It re-weights the ELBO to emphasise larger $t$ (which need more accurate denoising) and turns out to be the right perceptual prior.

### 4.11 The DDPM training algorithm (slide 67)

```
repeat:
    sample x0 ~ p_data
    sample t  ~ Uniform({1,...,T})
    sample eps ~ N(0, I)
    x_t = sqrt(α̂_t) x0 + sqrt(1 - α̂_t) eps
    take a gradient step on || eps - eps_θ(x_t, t) ||²
```

That's the entire training loop. No alternation, no GAN equilibrium, no adversarial dynamics. Just a noise-prediction regression.


In [ ]:
# ── DDPM noise predictor + training ───────────────────────────────────────

class SinusoidalTimeEmb(nn.Module):
    """Sinusoidal timestep embedding (same idea as positional encoding in Week 7)."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t.unsqueeze(-1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class NoisePredictor(nn.Module):
    """MLP that predicts noise given (x_t, t)."""
    def __init__(self, data_dim=2, hidden_dim=128, time_dim=32):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(time_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, data_dim))
    def forward(self, x, t):
        t_emb = self.time_emb(t)     # (B, time_dim)
        return self.net(torch.cat([x, t_emb], dim=-1))  # (B, data_dim)

# Training
torch.manual_seed(42)
ddpm_model = NoisePredictor()
opt_ddpm = torch.optim.Adam(ddpm_model.parameters(), lr=1e-3)

for step in range(5000):
    idx = torch.randperm(len(data_2d))[:256]
    x0 = data_2d[idx]                                        # (B, 2)
    t = torch.randint(0, T_diffusion, (256,))                # (B,)
    eps = torch.randn_like(x0)                                # (B, 2)
    ab = alpha_bars[t].unsqueeze(-1)                          # (B, 1)
    xt = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * eps      # (B, 2)
    eps_pred = ddpm_model(xt, t.float())                      # (B, 2)
    loss = F.mse_loss(eps_pred, eps)
    opt_ddpm.zero_grad(); loss.backward(); opt_ddpm.step()
    if (step + 1) % 1000 == 0:
        print(f'  DDPM step {step+1}, loss: {loss.item():.4f}')

### 4.12 Sampling: the reverse Markov chain (slide 67)

Once trained, sampling iterates the learned reverse:

```
x_T ~ N(0, I)
for t = T, T-1, ..., 1:
    z ~ N(0, I) if t > 1 else 0
    x_{t-1} = (1/sqrt(α_t)) (x_t - β_t / sqrt(1 - α̂_t) * eps_θ(x_t, t))  +  σ_t z
return x_0
```

Two design choices live in this loop:
- **The mean update** is the ε-prediction reparameterisation we derived in §4.9.
- **The noise injection** $\sigma_t z$ has two common settings: $\sigma_t^2 = \beta_t$ (DDPM-A) or $\sigma_t^2 = \tilde\beta_t$ (DDPM-B). Ho et al. (2020) report that both work; they usually use $\sigma_t^2 = \beta_t$.


In [ ]:
# ── DDPM sampling ───────────────────────────────────────────────────────────

@torch.no_grad()
def ddpm_sample(model, n_samples, T, betas, alpha_bars):
    alphas = 1.0 - betas
    x = torch.randn(n_samples, 2)  # start from noise
    trajectory = [x.clone()]
    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.float)
        eps_pred = model(x, t_batch)
        ab = alpha_bars[t]
        a = alphas[t]
        b = betas[t]
        mu = (1 / math.sqrt(a)) * (x - (b / math.sqrt(1 - ab)) * eps_pred)
        if t > 0:
            sigma = math.sqrt(b)
            x = mu + sigma * torch.randn_like(x)
        else:
            x = mu
        if t % (T // 8) == 0:
            trajectory.append(x.clone())
    return x, trajectory

t0 = time.time()
ddpm_samples, ddpm_traj = ddpm_sample(ddpm_model, 2000, T_diffusion, betas, alpha_bars)
ddpm_time = time.time() - t0
print(f'DDPM sampling ({T_diffusion} steps): {ddpm_time:.2f}s')

In [ ]:
# ── DDPM results ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.3, label='Real')
axes[0].scatter(ddpm_samples[:, 0].numpy(), ddpm_samples[:, 1].numpy(), s=1, alpha=0.3, c='purple', label='DDPM')
axes[0].legend(); axes[0].set_title(f'DDPM Samples ({T_diffusion} steps)')
axes[0].set_xlim(-3.5, 3.5); axes[0].set_ylim(-3.5, 3.5); axes[0].set_aspect('equal')

# Denoising trajectory for a few samples
for j in range(5):
    traj_x = [ddpm_traj[i][j, 0].item() for i in range(len(ddpm_traj))]
    traj_y = [ddpm_traj[i][j, 1].item() for i in range(len(ddpm_traj))]
    axes[1].plot(traj_x, traj_y, '-o', markersize=2, alpha=0.6)
axes[1].set_title('Denoising Trajectories (noise → data)')
axes[1].set_xlim(-4, 4); axes[1].set_ylim(-4, 4); axes[1].set_aspect('equal')
plt.tight_layout(); plt.show()
print('DDPM produces high-quality samples -- all 8 modes covered! But requires many steps.')

### 4.13 Network architectures (slide 68)

For images, the noise predictor $\varepsilon_\theta$ is almost universally a **U-Net**:

- **Down path**: a stack of ResNet blocks with strided convolutions, halving the spatial resolution at each level.
- **Bottleneck**: ResNet + self-attention layers, capturing global context.
- **Up path**: ResNet blocks with transposed/strided convolutions, doubling resolution. **Skip connections** carry features from the matching level on the down path.

Two ingredients are diffusion-specific:
1. **Time embedding.** The timestep $t$ is converted to a sinusoidal positional embedding (the same trick as in Transformers, Week 7), then projected with an MLP and added/concatenated to feature maps inside every ResNet block. This tells the network *where in the noise schedule it is*.
2. **Self-attention layers** at low spatial resolutions (typically $16\times16$ and $8\times8$). These are essential — without them, the model can't capture long-range structure in the image.

In our 2-D toy we don't need any of this complexity; a small MLP with a sinusoidal time embedding is enough. But know that this is the architecture every real diffusion model uses.

### 4.14 Comparing generative models (slide 69)

A useful summary, paraphrasing Lilian Weng's 2021 blog post that the lecture cites:

| | Likelihood | Sampling | Sample quality | Training stability | Mode coverage |
|---|---|---|---|---|---|
| GAN | None | 1 step | Sharp | **Unstable** | **Poor** |
| VAE | ELBO | 1 step | **Blurry** | Stable | Good |
| Flow (RealNVP, Glow) | Exact | 1 step | OK | Stable | Good |
| Diffusion | ELBO bound | **Many steps** | **Excellent** | Stable | Excellent |

### 4.15 The generative trilemma (slide 70)

Xiao et al. (ICLR 2022) summarised the field as a **trilemma**: any generative model can give you at most **two** of:

- **High sample quality**
- **Fast sampling**
- **Mode coverage / diversity**

| Model | Quality | Speed | Diversity |
|---|---|---|---|
| GAN | ✓ | ✓ | ✗ |
| VAE | ✗ | ✓ | ✓ |
| DDPM | ✓ | ✗ | ✓ |

Parts 5–7 of this notebook can be read as **a research programme to defeat the trilemma** — get DDPM-quality samples in GAN-speed (1 step) without sacrificing diversity. Score-based / SDE (Part 5) sets up the maths; flow matching (Part 6) makes paths straighter; consistency models (Part 7) close the gap to one step.


---
## Part 5 — The Continuous View: Score Functions, SDEs, PF-ODE
*(Lecture slides 71–98)*

### 5.1 Why a continuous view?

DDPM defines a discrete chain of $T = 1000$ steps. This is fine but it raises a question: what *is* the chain doing in the limit $T \to \infty$ with $\beta_t \to 0$? The answer turns out to be a **stochastic differential equation (SDE)**. Reframing diffusion as an SDE has three concrete payoffs:

1. **All known noise schedules are special cases** of one continuous SDE family. This unifies VP-SDE, VE-SDE, sub-VP, etc.
2. The score function $\nabla_x \log p_t(x)$ becomes the *single object* you have to learn — denoising, noise prediction, $x_0$-prediction are all parameterisations of the same thing.
3. The reverse SDE has a **deterministic twin** called the Probability Flow ODE (PF-ODE). The PF-ODE has the same marginals but is differentiable, invertible, gives tractable likelihoods, and lets you use 100× fewer steps.

This part develops all three.

### 5.2 The score function (slide 73)

The **score** of a distribution $p$ at a point $x$ is the gradient of the log-density:

$$s(x) \;=\; \nabla_x \log p(x).$$

Geometrically it is a vector field that points toward higher probability. For a Gaussian $\mathcal N(\mu, \sigma^2 I)$,

$$s(x) \;=\; -\frac{x - \mu}{\sigma^2}\quad \text{— an arrow pointing at }\mu.$$

For a mixture of Gaussians, it points toward the *nearest* mode, weighted by responsibilities. We will visualise this in a moment.

A **score-based model** $s_\theta(x)$ explicitly approximates $s(x)$. If we have it, we can sample from $p$ via Langevin dynamics or an SDE (we'll see how).


In [ ]:
# ── Score field visualisation ────────────────────────────────────────────────
# Compute the score of the 8-Gaussian mixture analytically
def score_8gaussians(x, std=0.05):
    """Analytic score of 8-Gaussian mixture."""
    angles = torch.linspace(0, 2 * math.pi, 9)[:-1]
    centres = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 2.0  # (8, 2)
    # Compute unnormalised density contributions
    diffs = x.unsqueeze(1) - centres.unsqueeze(0)  # (N, 8, 2)
    log_ps = -0.5 * (diffs ** 2).sum(-1) / std**2  # (N, 8)
    weights = F.softmax(log_ps, dim=1)               # (N, 8)
    # Weighted sum of per-component scores
    scores = -(weights.unsqueeze(-1) * diffs).sum(1) / std**2  # (N, 2)
    return scores

# Grid
grid_x = torch.linspace(-3.5, 3.5, 20)
grid_y = torch.linspace(-3.5, 3.5, 20)
xx, yy = torch.meshgrid(grid_x, grid_y, indexing='xy')
grid_pts = torch.stack([xx.flatten(), yy.flatten()], dim=1)
scores = score_8gaussians(grid_pts)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.1, c='gray')
ax.quiver(grid_pts[:, 0].numpy(), grid_pts[:, 1].numpy(),
          scores[:, 0].numpy(), scores[:, 1].numpy(),
          color='blue', alpha=0.7, scale=200)
ax.set_title('Score Field: arrows point toward data modes', fontsize=12)
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

**Read the field.** The arrows are exactly the gradients the network would have to learn. They are well defined at high density (near the modes) but become *ill-conditioned in the gaps* between modes — this is one of the failure modes of naive score matching, and is exactly what the noise schedule fixes (by smoothing $p_t$ at large $t$ until the gaps disappear).

### 5.3 From DDPM to an SDE (slides 74–80)

Recall the DDPM forward step: $x_t = \sqrt{1 - \beta_t}\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon_t$. Rearrange:

$$x_t - x_{t-1} \;=\; \big(\sqrt{1-\beta_t} - 1\big)\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon_t.$$

For small $\beta_t$, $\sqrt{1-\beta_t} - 1 \approx -\tfrac12 \beta_t$, so

$$x_t - x_{t-1} \;\approx\; -\tfrac12 \beta_t\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon_t.$$

Let $\Delta t = 1/T$ and $\beta_t = \bar\beta(t)\,\Delta t$ for a continuous schedule $\bar\beta:[0,1]\to\mathbb R_{>0}$. Then in the limit $\Delta t \to 0$ we recognise the SDE

$$\boxed{\;dx \;=\; -\tfrac12\,\bar\beta(t)\,x\,dt \;+\; \sqrt{\bar\beta(t)}\,dw\;}$$

This is the **Variance-Preserving SDE (VP-SDE)** of Song et al. (ICLR 2021). The two terms have names:
- **Drift** $f(x,t) = -\tfrac12 \bar\beta(t)\,x$: pulls $x$ toward zero (the mode of the prior).
- **Diffusion** $g(t) = \sqrt{\bar\beta(t)}$: injects Gaussian noise.

### 5.4 Crash course in differential equations (slide 79)

| | ODE | SDE |
|---|---|---|
| **Form** | $dx = f(x,t)\,dt$ | $dx = f(x,t)\,dt + g(t)\,dw$ |
| **Trajectories** | Deterministic | Stochastic |
| **Numerical solver** | Euler, Runge–Kutta | Euler–Maruyama, stochastic Heun |

Here $w$ is **Brownian motion** — a Gaussian process with independent zero-mean increments $w_{t+\Delta t} - w_t \sim \mathcal N(0, \Delta t I)$. The Euler–Maruyama discretisation of an SDE is

$$x_{t + \Delta t} \;\approx\; x_t \;+\; f(x_t, t)\,\Delta t \;+\; g(t)\,\sqrt{\Delta t}\,\varepsilon,\qquad \varepsilon \sim \mathcal N(0, I).$$

If $g(t) = 0$ this collapses to standard Euler.

### 5.5 The reverse-time SDE (slides 82–83)

Anderson (1982) proved a remarkable fact: a forward-time SDE with the marginals $\{p_t\}$ admits a **reverse-time SDE** that has the *same marginals* (running time backwards):

$$\boxed{\;dx \;=\; \left[f(x,t) - g(t)^2\,\nabla_x \log p_t(x)\right]dt \;+\; g(t)\,d\bar w\;}$$

where $d\bar w$ is reverse-time Brownian motion. Crucially, **the score $\nabla_x \log p_t(x)$ shows up in the reverse drift**.

So if we knew the marginal score $\nabla_x \log p_t(x)$ at every time and place, we could simulate the reverse SDE backwards from $t = 1$ (pure noise) to $t = 0$ (data) and generate samples.

### 5.6 But we don't have the score: score matching (slides 85–86)

The marginal score is not known in closed form because $p_t(x) = \int p_{\text{data}}(y)\,q_t(x \mid y)\,dy$ is an intractable integral. **Naive score matching** — directly regress a network on $\nabla_x \log p_t(x)$ — is impossible because we don't have the targets.

There is a beautiful trick due to **Vincent (2011)**: **denoising score matching**. Instead of regressing on the marginal score, regress on the *conditional* score $\nabla_{x_t} \log q_t(x_t \mid x_0)$, which **is** tractable.

#### The Vincent identity

Define the score-matching loss

$$J(\theta) \;=\; \tfrac12\,\mathbb E_{p_t(x_t)}\!\left[\|s_\theta(x_t, t) - \nabla_{x_t} \log p_t(x_t)\|^2\right].$$

Vincent showed that, up to a constant in $\theta$,

$$J(\theta) \;=\; \tfrac12\,\mathbb E_{p_{\text{data}}(x_0)}\;\mathbb E_{q_t(x_t\mid x_0)}\!\left[\|s_\theta(x_t, t) - \nabla_{x_t}\log q_t(x_t\mid x_0)\|^2\right] + C.$$

The proof is a one-page exercise: expand the square, use $p_t(x_t) = \int p_{\text{data}}(x_0) q_t(x_t\mid x_0)\,dx_0$, and apply the chain rule to push the gradient inside. The right-hand side is **tractable** because $q_t(x_t \mid x_0)$ is a Gaussian and its conditional score has a closed form.

#### Conditional score for VP-SDE

For $q_t(x_t\mid x_0) = \mathcal N(\sqrt{\bar\alpha_t}\,x_0,\,(1-\bar\alpha_t)I)$,

$$\nabla_{x_t}\log q_t(x_t\mid x_0) \;=\; -\frac{x_t - \sqrt{\bar\alpha_t}\,x_0}{1 - \bar\alpha_t} \;=\; -\frac{\varepsilon}{\sqrt{1-\bar\alpha_t}}.$$

So the score is just $-\varepsilon$, scaled.

### 5.7 ε-prediction is denoising score matching (slide 87)

Combining the previous two:

$$s_\theta(x_t, t) \;=\; -\frac{\varepsilon_\theta(x_t, t)}{\sqrt{1-\bar\alpha_t}}.$$

This is the famous identity: **a DDPM noise predictor $\varepsilon_\theta$ and a score model $s_\theta$ are the same network up to a multiplicative scalar**. Anything you can do with one, you can do with the other. The two communities (Ho et al.'s "noise prediction" and Song et al.'s "score-based") were doing the same thing, just with different language.

> **One more parameterisation.** $x_0$-prediction is also equivalent: $\hat x_0(x_t, t) = \frac{1}{\sqrt{\bar\alpha_t}}(x_t - \sqrt{1-\bar\alpha_t}\,\varepsilon_\theta(x_t, t))$. So the four common parameterisations — score $s_\theta$, noise $\varepsilon_\theta$, $x_0$-prediction, v-prediction $v_\theta = \alpha_t \varepsilon - \sigma_t x_0$ (Salimans & Ho 2022) — are all linear combinations of each other. Real implementations (EDM, Imagen, SD3) often use v-prediction because its loss is best balanced across timesteps.

### 5.8 Loss weightings (slide 88)

The denoising score-matching loss with general weighting is

$$\mathcal L(\theta) \;=\; \mathbb E_{t,\,x_0,\,\varepsilon}\!\left[\,\lambda(t)\,\|s_\theta(x_t, t) - \nabla_{x_t}\log q_t(x_t\mid x_0)\|^2\,\right].$$

Different choices of $\lambda(t)$ give different objectives:

| $\lambda(t)$ | Equivalent to |
|---|---|
| $1 / \sigma_t^2$ | The exact ELBO term $L_{t-1}$ — gives **maximum likelihood** |
| $1$ | Ho et al.'s $L_{\text{simple}}$ — gives **best perceptual quality** |
| $\sigma_t / \alpha_t$ | EDM-style preconditioning — best modern default |

The trade-off is well known: optimising for likelihood gives blurry samples, optimising for $L_{\text{simple}}$ gives sharp samples but a worse bits-per-dimension number. Slide 88 visualises this trade-off.

### 5.9 Limitations of the SDE view (slide 89)

The reverse SDE works, but it has three drawbacks:

1. **Stochastic trajectories** — generation is non-deterministic, which makes inversion / editing hard.
2. **No tractable likelihood** — you can't compute $\log p_\theta(x)$ from the SDE directly.
3. **Inefficient sampling** — to numerically solve the SDE accurately you typically need $\sim 1000$ steps.

The next subsection fixes all three with a single trick: replace the SDE with an equivalent ODE.

### 5.10 The Probability Flow ODE (slides 90–94)

Song et al. (2021) prove that the SDE

$$dx = f(x,t)\,dt + g(t)\,dw$$

has the **same marginals** at every $t$ as the deterministic ODE

$$\boxed{\;dx \;=\; \left[f(x,t) - \tfrac12\,g(t)^2\,\nabla_x \log p_t(x)\right] dt\;}$$

(no Brownian term). This is the **Probability Flow ODE (PF-ODE)**. The proof uses the Fokker–Planck equation: write down the PDE that $p_t$ satisfies under the SDE, then check that the same PDE is also generated by the ODE. (You can find a one-page version in Appendix D.1 of Song et al.)

Substituting the score $\nabla_x \log p_t = -\varepsilon_\theta / \sqrt{1-\bar\alpha_t}$, the PF-ODE in noise-prediction form for VP-SDE is

$$\frac{dx}{dt} \;=\; -\tfrac12\,\bar\beta(t)\,x \;+\; \tfrac12\,\bar\beta(t)\,\frac{\varepsilon_\theta(x, t)}{\sqrt{1-\bar\alpha_t}}.$$

This is the velocity field that DDIM, EDM, and most modern fast samplers numerically integrate.

### 5.11 What the PF-ODE buys you

Three things, simultaneously:

1. **Deterministic generation.** Same noise → same image. This unlocks deterministic encoding ($x_0 \to x_T$ by running the ODE forwards) and editing (perturb $x_T$, decode).
2. **Tractable likelihood (slide 94).** A continuous-time ODE is a *neural ODE*, equivalently a *continuous normalising flow*. The instantaneous change-of-variables formula gives
   $$\log p_0(x_0) \;=\; \log p_T(x_T) \;+\; \int_0^T \mathrm{tr}\!\left(\frac{\partial v_\theta(x_t, t)}{\partial x_t}\right) dt.$$
   The trace is estimated via Hutchinson's trick. This is how diffusion model bits-per-dimension numbers are computed.
3. **Faster sampling.** Because the ODE is smooth, you can use any black-box ODE solver (Runge–Kutta, DPM-Solver, exponential integrators, …) and get away with 10–50 steps instead of 1000.

### 5.12 Numerical solvers (slide 95)

| Equation | Naive solver | Modern solver |
|---|---|---|
| Forward/reverse SDE | Euler–Maruyama | Stochastic Heun, Predictor-Corrector |
| PF-ODE | Euler | DPM-Solver, DPM-Solver++, UniPC, Heun, RK4, LMS |

Here is the key idea behind **DPM-Solver** (Lu et al. 2022), the most influential fast sampler: the PF-ODE drift can be split into a *linear* part (the drift $f(x,t)$) and a *non-linear* part (the score $\varepsilon_\theta$). The linear part can be integrated **exactly** in closed form; only the non-linear part needs numerical approximation. This gives much smaller error per step. With DPM-Solver-3 you can get high-quality samples in 10–20 NFE.

### 5.13 SDE vs ODE: pros and cons (slide 96)

| | SDE | ODE |
|---|---|---|
| Determinism | Stochastic | Deterministic |
| Error correction | Noise re-injection compensates for solver errors | Errors compound — need higher-order solver |
| Sampling speed | Slower (need fine steps) | Faster (high-order solvers) |
| Likelihood | None | Yes (CNF) |
| Inversion / editing | Hard | Easy |
| Best for | Maximum sample quality at NFE → ∞ | Fast sampling, editing |

Empirically, very high-quality samples often use a *hybrid*: ODE for most of the trajectory, with occasional noise injection (Karras et al., EDM, 2022).

### 5.14 Unique identifiability (slide 97)

A subtle but important fact: **the optimal score $\nabla_x \log p_t(x)$ is uniquely determined by $p_{\text{data}}$ and the forward process**. So under ideal training (infinite data, infinite capacity), every diffusion model converges to the *same* score function — there is no architecture-specific bias in the "right answer". Different model families, datasets, and training tricks differ only in **how close they get to this unique score**.

Compare with GANs, which have multiple equilibria and can converge to *different* generators that all minimise the JS divergence.

### 5.15 Why use the differential equation framework? (slide 98)

To summarise:
- Decades of literature on **fast SDE/ODE solvers** become directly applicable.
- A single continuous-time view **unifies** DDPM, NCSN, smld, VP, VE, sub-VP, etc.
- **Deterministic encoding and likelihood** come for free via the PF-ODE.
- The framework gives a **clean conceptual picture** of what diffusion models are doing — they are learning a vector field that transports noise to data.

The next two cells show this picture explicitly: ODE trajectories are smooth and deterministic, SDE trajectories are jagged and stochastic, but they sample from the same distribution.


In [ ]:
# ── ODE vs SDE trajectory comparison ─────────────────────────────────────
# Simulate SDE (stochastic) vs ODE (deterministic) paths using trained DDPM
torch.manual_seed(123)
n_show = 8
x_start = torch.randn(n_show, 2) * 2.5

def simulate_reverse(model, x_init, T, alpha_bars, betas, stochastic=True, n_steps=200):
    step_size = max(1, T // n_steps)
    timesteps = list(range(T - 1, -1, -step_size))[:n_steps]
    x = x_init.clone()
    path = [x.clone()]
    for t in timesteps:
        t_b = torch.full((x.size(0),), t, dtype=torch.float)
        eps = model(x, t_b)
        ab = alpha_bars[t]
        a = 1.0 - betas[t]
        mu = (1/math.sqrt(a)) * (x - (betas[t]/math.sqrt(1-ab)) * eps)
        if stochastic and t > 0:
            x = mu + math.sqrt(betas[t]) * torch.randn_like(x)
        else:
            x = mu
        path.append(x.clone())
    return path

with torch.no_grad():
    sde_path = simulate_reverse(ddpm_model, x_start, T_diffusion, alpha_bars, betas, stochastic=True)
    ode_path = simulate_reverse(ddpm_model, x_start, T_diffusion, alpha_bars, betas, stochastic=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for j in range(n_show):
    sx = [sde_path[i][j,0].item() for i in range(len(sde_path))]
    sy = [sde_path[i][j,1].item() for i in range(len(sde_path))]
    ax1.plot(sx, sy, alpha=0.5)
ax1.scatter(data_2d[:1000,0], data_2d[:1000,1], s=1, alpha=0.05, c='gray')
ax1.set_title('SDE Trajectories (stochastic, noisy paths)'); ax1.set_xlim(-4,4); ax1.set_ylim(-4,4); ax1.set_aspect('equal')

for j in range(n_show):
    ox = [ode_path[i][j,0].item() for i in range(len(ode_path))]
    oy = [ode_path[i][j,1].item() for i in range(len(ode_path))]
    ax2.plot(ox, oy, alpha=0.5)
ax2.scatter(data_2d[:1000,0], data_2d[:1000,1], s=1, alpha=0.05, c='gray')
ax2.set_title('ODE Trajectories (deterministic, smooth paths)'); ax2.set_xlim(-4,4); ax2.set_ylim(-4,4); ax2.set_aspect('equal')
plt.tight_layout(); plt.show()
print('Both converge to data modes. ODE paths are smooth but CURVED → many steps needed.')

### 5.16 DDIM as a discrete PF-ODE solver

DDIM (Song et al. ICLR 2021) was actually invented *before* the PF-ODE was published — but in hindsight it is **exactly Euler's method on the PF-ODE** with the integration variable rewritten in terms of $\bar\alpha_t$ rather than $t$. The DDIM update rule (with $\eta = 0$) is

$$x_{t-1} \;=\; \sqrt{\bar\alpha_{t-1}}\;\underbrace{\left(\frac{x_t - \sqrt{1-\bar\alpha_t}\,\varepsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}\right)}_{\hat x_0}\;+\;\sqrt{1-\bar\alpha_{t-1}}\;\varepsilon_\theta(x_t,t).$$

Read it as: "estimate $x_0$ from $x_t$, then jump back to $x_{t-1}$ along the *same* ε direction". The estimated $\hat x_0$ stays consistent across timesteps (deterministic), so you can subsample timesteps freely without retraining the model.

**Why this matters.**
- The same trained DDPM can be sampled in 1000 steps (full DDPM), 100 steps (DDIM, $S=100$), or 50 steps (DDIM, $S=50$) — the model parameters are unchanged.
- DDIM with $\eta > 0$ smoothly interpolates between deterministic ($\eta=0$, ODE) and stochastic ($\eta=1$, the full DDPM SDE).
- DDIM is the *first half* of the speedup story. Its successors (DPM-Solver, EDM-Heun, UniPC) are higher-order ODE integrators on the same PF-ODE.


In [ ]:
# ── DDIM sampling ───────────────────────────────────────────────────────────

@torch.no_grad()
def ddim_sample(model, n_samples, T, alpha_bars, num_steps=50, eta=0.0):
    """DDIM sampling with S << T steps. eta=0 is deterministic."""
    # Create sub-sequence of timesteps
    step_size = T // num_steps
    timesteps = list(range(T - 1, -1, -step_size))[:num_steps]
    x = torch.randn(n_samples, 2)
    for i, t in enumerate(timesteps):
        t_batch = torch.full((n_samples,), t, dtype=torch.float)
        eps_pred = model(x, t_batch)
        ab_t = alpha_bars[t]
        ab_prev = alpha_bars[timesteps[i + 1]] if i + 1 < len(timesteps) else torch.tensor(1.0)
        # Predicted x_0
        x0_pred = (x - math.sqrt(1 - ab_t) * eps_pred) / math.sqrt(ab_t)
        # DDIM update
        sigma = eta * math.sqrt((1 - ab_prev) / (1 - ab_t)) * math.sqrt(1 - ab_t / ab_prev)
        dir_xt = math.sqrt(1 - ab_prev - sigma**2) * eps_pred
        x = math.sqrt(ab_prev) * x0_pred + dir_xt
        if sigma > 0:
            x = x + sigma * torch.randn_like(x)
    return x

# Compare different step counts
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
configs = [('DDPM', T_diffusion), ('DDIM-50', 50), ('DDIM-20', 20), ('DDIM-10', 10)]
times_compare = []
for ax, (name, steps) in zip(axes, configs):
    t0 = time.time()
    if name == 'DDPM':
        samples, _ = ddpm_sample(ddpm_model, 2000, T_diffusion, betas, alpha_bars)
    else:
        samples = ddim_sample(ddpm_model, 2000, T_diffusion, alpha_bars, num_steps=steps)
    elapsed = time.time() - t0
    times_compare.append((name, elapsed))
    ax.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=1, alpha=0.3)
    ax.set_title(f'{name}\n{elapsed:.2f}s'); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.suptitle('Same model, different sampling speeds', fontsize=13)
plt.tight_layout(); plt.show()
for name, t in times_compare: print(f'  {name}: {t:.3f}s')

### 5.17 Summary of Part 5

- The DDPM forward chain is the discretisation of an SDE.
- The reverse SDE depends on the **score function** $\nabla_x \log p_t(x)$.
- **Denoising score matching** (Vincent 2011) lets us train a network on the score without ever knowing the marginal — by regressing on the conditional score, which is closed-form.
- ε-prediction, score prediction, $x_0$-prediction, v-prediction are **the same network** up to linear transformations.
- The reverse SDE has a **deterministic twin**, the **Probability Flow ODE**, with the same marginals.
- The PF-ODE makes generation faster, deterministic, invertible, and likelihood-tractable.
- DDIM is exactly Euler's method on the PF-ODE.

This is the conceptual peak of the lecture — Parts 6 and 7 build on this framework.


---
## Part 6 — The Transport View: Flow Matching and Rectified Flow
*(Lecture slides 99–108)*

### 6.1 Why move beyond the score SDE? (slide 100)

The PF-ODE was a big win, but the lecture identifies three remaining problems:

1. **Curved trajectories.** The PF-ODE paths from noise to data are *curved* — they trace out the posterior $\mathbb E[x_0 \mid x_t]$, which bends. Curved paths need many small Euler steps.
2. **Fixed noise schedule.** You have to *design* $f(x,t)$ and $g(t)$ carefully to get a good $p_T \approx \mathcal N(0, I)$. There is no principled reason that this particular drift / diffusion is optimal.
3. **Indirect training signal.** You learn the score, then plug it into a velocity field. Why not just learn the velocity field directly?

**Flow matching** (Lipman et al., ICLR 2023) takes the third question seriously: instead of learning a score and deriving a velocity, **learn the velocity directly**.

### 6.2 The velocity field viewpoint (slide 101)

Define a continuous-time map from a base distribution $p_0$ at $t=0$ to a target distribution $p_1 = p_{\text{data}}$ at $t=1$ via an ODE

$$\frac{dx}{dt} \;=\; v_\theta(x, t),\qquad x(0) \sim p_0 = \mathcal N(0, I).$$

The velocity field $v_\theta(x, t)$ specifies *how each point moves* through space. If we get $v_\theta$ right, then solving this ODE from $t=0$ to $t=1$ takes a noise sample to a data sample.

This is the same maths as **continuous normalising flows** (Chen et al. 2018) — but the training objective is different.

### 6.3 Goal: marginal flow matching, intractable (slide 102)

Suppose we have a *target* time-varying density $p_t$ (interpolating from $\mathcal N(0,I)$ to $p_{\text{data}}$) and a *target* velocity field $u_t$ that generates it. Then we'd like to match our model:

$$\mathcal L_{\text{FM}}(\theta) \;=\; \mathbb E_{t \sim U[0,1],\,x \sim p_t}\!\left[\|v_\theta(x, t) - u_t(x)\|^2\right].$$

The problem: we don't have access to $u_t(x)$ in closed form, because it depends on the marginal $p_t$ which is an integral over data.

### 6.4 Conditional Flow Matching: the trick (slide 103)

Define the velocity field **conditioned on a single data point** $x_1$. For each $x_1$, choose a **conditional path** $p_t(x \mid x_1)$ — a distribution that interpolates from $\mathcal N(0,I)$ at $t=0$ to a delta at $x_1$ at $t=1$ — and the *unique* velocity $u_t(x \mid x_1)$ that pushes mass along it.

The **Conditional Flow Matching (CFM)** loss is

$$\mathcal L_{\text{CFM}}(\theta) \;=\; \mathbb E_{t,\,x_1 \sim p_{\text{data}},\,x \sim p_t(\cdot\mid x_1)}\!\left[\|v_\theta(x, t) - u_t(x \mid x_1)\|^2\right].$$

The remarkable theorem of Lipman et al. (2023): **the CFM loss has the same gradient w.r.t. $\theta$ as the marginal FM loss** (the constants and irrelevant terms drop out). So we can train on the conditional loss — which is tractable — and learn the marginal velocity field.

### 6.5 The OT path: straight conditional trajectories

The simplest choice of conditional path is a **straight line** between a noise sample $x_0 \sim \mathcal N(0,I)$ and the data point $x_1$:

$$x_t \;=\; (1 - t)\,x_0 + t\,x_1,\qquad t \in [0,1].$$

Then the conditional velocity (just differentiate $x_t$ w.r.t. $t$) is

$$u_t(x_t \mid x_0, x_1) \;=\; x_1 - x_0.$$

Substituting into CFM:

$$\boxed{\;\mathcal L_{\text{CFM}}^{\text{OT}}(\theta) \;=\; \mathbb E_{t,\,x_0 \sim \mathcal N(0,I),\,x_1 \sim p_{\text{data}}}\!\left[\,\big\|v_\theta((1-t)x_0 + tx_1,\;t) - (x_1 - x_0)\big\|^2\,\right]\;}$$

That's it. The *entire* training loop is: sample $t$, sample noise $x_0$, sample data $x_1$, interpolate, regress on $x_1 - x_0$. No SDE, no score, no noise schedule design. This is what the lecture calls **OT path Flow Matching**, because the straight line is the optimal-transport plan when both endpoints are individual samples.

> **Connection to diffusion.** The VP path used by DDPM is *also* a special case of CFM — it just uses a different conditional path, namely a Gaussian one whose mean and variance follow the noise schedule. So flow matching **strictly generalises** diffusion: any DDPM is a CFM with a particular path choice.

### 6.6 Code: Flow Matching on the 8-Gaussians toy


In [ ]:
# ── Flow Matching: velocity field + training ─────────────────────────────

class VelocityField(nn.Module):
    """Predict velocity v(x_t, t) for flow matching."""
    def __init__(self, data_dim=2, hidden_dim=128, time_dim=32):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(time_dim)  # reuse from Phase 2
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, data_dim))
    def forward(self, x, t):
        t_emb = self.time_emb(t * 100)  # scale t to match timestep range
        return self.net(torch.cat([x, t_emb], dim=-1))

# Training
torch.manual_seed(42)
fm_model = VelocityField()
opt_fm = torch.optim.Adam(fm_model.parameters(), lr=1e-3)

for step in range(5000):
    idx = torch.randperm(len(data_2d))[:256]
    x1 = data_2d[idx]                          # data points
    x0 = torch.randn_like(x1)                  # noise
    t = torch.rand(256)                         # t ~ U(0, 1)
    xt = (1 - t.unsqueeze(-1)) * x0 + t.unsqueeze(-1) * x1  # straight-line interpolation
    target = x1 - x0                            # target velocity
    v_pred = fm_model(xt, t)
    loss = F.mse_loss(v_pred, target)
    opt_fm.zero_grad(); loss.backward(); opt_fm.step()
    if (step + 1) % 1000 == 0:
        print(f'  Flow Matching step {step+1}, loss: {loss.item():.4f}')

In [ ]:
# ── Flow Matching: Euler ODE sampling ─────────────────────────────────────

@torch.no_grad()
def fm_sample(model, n_samples, num_steps=100):
    """Sample by solving dx/dt = v(x,t) from t=0 to t=1 via Euler."""
    x = torch.randn(n_samples, 2)
    dt = 1.0 / num_steps
    trajectory = [x.clone()]
    for i in range(num_steps):
        t = torch.full((n_samples,), i * dt)
        x = x + model(x, t) * dt
        if i % (num_steps // 8) == 0:
            trajectory.append(x.clone())
    trajectory.append(x.clone())
    return x, trajectory

t0 = time.time()
fm_samples, fm_traj = fm_sample(fm_model, 2000, num_steps=50)
fm_time = time.time() - t0
print(f'Flow Matching sampling (50 steps): {fm_time:.2f}s')

In [ ]:
# ── Flow Matching visualisation ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Flow trajectories
for j in range(12):
    fx = [fm_traj[i][j, 0].item() for i in range(len(fm_traj))]
    fy = [fm_traj[i][j, 1].item() for i in range(len(fm_traj))]
    ax1.plot(fx, fy, '-o', markersize=2, alpha=0.5)
ax1.scatter(data_2d[:1000, 0], data_2d[:1000, 1], s=1, alpha=0.05, c='gray')
ax1.set_title('Flow Matching Trajectories (approximately STRAIGHT!)')
ax1.set_xlim(-4, 4); ax1.set_ylim(-4, 4); ax1.set_aspect('equal')

# Samples
ax2.scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.2, label='Real')
ax2.scatter(fm_samples[:, 0].numpy(), fm_samples[:, 1].numpy(), s=1, alpha=0.2, c='green', label='FM')
ax2.legend(); ax2.set_title('Flow Matching Samples (50 steps)')
ax2.set_xlim(-3.5, 3.5); ax2.set_ylim(-3.5, 3.5); ax2.set_aspect('equal')
plt.tight_layout(); plt.show()
print('Paths are much straighter than diffusion ODE paths → fewer steps needed!')

**Read the trajectories.** They should be visibly **straighter** than the DDIM trajectories from Part 5 — many of them look almost like straight lines from a noise point to a data point. That's exactly what CFM with the OT path is supposed to give.

### 6.7 Step count vs quality

Because the paths are straighter, fewer Euler steps are needed to integrate them accurately. The next cell sweeps the number of steps and shows the trade-off.


In [ ]:
# ── Step count vs quality comparison ───────────────────────────────────────
step_counts = [1, 5, 10, 20, 50, 100]
fig, axes = plt.subplots(1, len(step_counts), figsize=(18, 3))
for ax, s in zip(axes, step_counts):
    samples, _ = fm_sample(fm_model, 1000, num_steps=s)
    ax.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=1, alpha=0.3)
    ax.set_title(f'{s} step{"s" if s > 1 else ""}')
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.suptitle('Flow Matching: Quality vs Number of Euler Steps', fontsize=13)
plt.tight_layout(); plt.show()
print('Flow Matching degrades gracefully -- even 10 steps gives recognisable modes.')

You should see that even **5–10 steps** give recognisable samples, which is impossible for vanilla DDPM and only marginal for DDIM. This is the speed payoff of the transport view.

### 6.8 Rectified Flow: motivation (slide 104)

Even with OT-CFM there is a subtle problem: the **conditional** paths (one per noise/data pair) are straight, but the **marginal** velocity field can still curve. Why?

> Different $(x_0, x_1)$ pairs can produce trajectories that **cross** each other. At a crossing point, two different "true" velocities meet — and the network has to predict their *average*, which is neither. Average of two crossing straight lines is a curve.

This is the crucial insight of **Rectified Flow** (Liu et al., ICLR 2023): the marginal flow is curved *because of crossings in the coupling*, not because of the noise schedule.

### 6.9 The reflow procedure (slide 105–106)

Rectified Flow straightens the marginal flow with an iterative procedure:

1. **Coupling.** Sample independent pairs $(x_0^{(i)}, x_1^{(i)})$ — random noise paired with random data. This is what plain CFM uses.
2. **Train $v_\theta^{(0)}$** with CFM on these pairs.
3. **Generate new couplings** by running the trained ODE: for each $x_0$, integrate $v_\theta^{(0)}$ from 0 to 1 and let $\hat x_1$ be the result. The new coupling is $(x_0, \hat x_1)$ — but now the noise and the produced sample are **causally linked**, not independent.
4. **Retrain $v_\theta^{(1)}$** with CFM on these new couplings. Because each $x_0$ now consistently maps to a particular $\hat x_1$, fewer trajectories cross each other, and the marginal velocity field is straighter.
5. **Repeat.**

The lecture writes this as $v^0 \to \text{reflow} \to v^1 \to \text{reflow} \to v^2 \to \cdots$. Each reflow makes the field straighter. After 1–2 reflows the resulting flow is straight enough that **a single Euler step** at $t = 0.5$ recovers reasonable samples — which is the foundation of one-step image generators like Stable Diffusion 3 (Esser et al. 2024) and Flux.

> **Why does straightness imply few steps?** A perfectly straight ODE has a constant velocity along each trajectory, so a single Euler step is *exact* — no discretisation error at all. Curved trajectories have changing velocity, so Euler accumulates error and you need finer steps to compensate.

### 6.10 Summary of Part 6 (slide 107)

- **Flow Matching** trains a velocity field directly, with a simulation-free MSE loss.
- **Conditional FM** makes this tractable by replacing the marginal target with a conditional one (proved equivalent in gradient).
- The **OT path** gives straight conditional trajectories and a remarkably simple training objective: regress on $x_1 - x_0$.
- **Rectified Flow** iteratively straightens the *marginal* flow by reflowing on coupled pairs, recovering single-step generation in the limit.
- Diffusion is a *special case* of flow matching with a Gaussian conditional path.

Where this leaves the trilemma: with rectified flow we are at "5 steps" and "high quality" — almost back to GAN speed, with diffusion-grade quality and diversity. The last step (one shot) needs one more idea, which is Part 7.


---
## Part 7 — Efficiency Frontier: Consistency Models
*(Lecture slides 109–121)*

### 7.1 The speed–quality trade-off so far (slide 110)

| Method | Network function evaluations (NFE) for high quality |
|---|---|
| GAN | 1 |
| DDPM | ~1000 |
| DDIM | ~50 |
| DPM-Solver | ~10–20 |
| Flow Matching (CFM) | ~10–50 |
| Rectified Flow (1 reflow) | ~5 |
| **Consistency models (this section)** | **1** |

The **generative learning trilemma** said you can have at most two of {quality, speed, diversity}. The story of Parts 5–7 is the systematic dismantling of that claim. Consistency models complete it: they recover **single-step sampling** while keeping diffusion-grade quality and full mode coverage.

### 7.2 The core idea (slides 111–112)

Recall that the PF-ODE defines, for every starting point $x$, a unique deterministic trajectory $\{x_t\}_{t \in [0,T]}$ from $x_T \sim \mathcal N(0,\sigma_T^2 I)$ to $x_0 \in p_{\text{data}}$.

A **consistency function** is a map

$$f_\theta(x_t, t) \;\to\; x_0$$

that, given any point $x_t$ along the trajectory, returns the *origin* (the clean data point) of that trajectory. By construction, **all points on the same trajectory map to the same $x_0$** — that is the "consistency" property.

If we have such an $f_\theta$, sampling is **one network call**:
1. Sample $x_T \sim \mathcal N(0, \sigma_T^2 I)$.
2. Return $f_\theta(x_T, T)$.

That's it.

### 7.3 The two defining constraints (slide 113)

Consistency models are defined by two constraints that $f_\theta$ must satisfy:

#### Boundary condition

$$f_\theta(x_0, 0) \;=\; x_0\quad\text{for all } x_0.$$

At $t = 0$ the function must be the identity. (Otherwise a clean $x_0$ would not map to itself.)

#### Self-consistency

For all $t, t' \in [0, T]$ and all $(x_t, x_{t'})$ on the **same** PF-ODE trajectory,

$$f_\theta(x_t, t) \;=\; f_\theta(x_{t'}, t').$$

Both constraints have to hold *simultaneously*. The cleverness of consistency models is **how each one is enforced**:
- The **boundary condition** is enforced **by parameterisation** (an architectural skip connection) — built into the function class so it's true by construction.
- **Self-consistency** is enforced **by training** — a loss term that penalises pairs of points on the same trajectory mapping to different outputs.

### 7.4 Architectural enforcement of the boundary (slide 116)

You'd like $f_\theta(x_t, t)$ to be "approximately $x_t$ when $t$ is small" and "approximately the prediction of a deep network when $t$ is large". This is exactly a **skip connection with time-dependent weights**:

$$f_\theta(x, t) \;=\; c_{\text{skip}}(t)\,x \;+\; c_{\text{out}}(t)\,F_\theta(x, t),$$

where $F_\theta$ is the underlying neural network, and the scalars satisfy

$$c_{\text{skip}}(0) = 1,\qquad c_{\text{out}}(0) = 0.$$

At $t = 0$ this collapses to $f_\theta(x_0, 0) = x_0$, **regardless of what $F_\theta$ outputs**. The boundary condition is *baked in*.

The same parameterisation form is used by EDM, v-prediction, and many other modern diffusion models — only the choice of $c_{\text{skip}}$ and $c_{\text{out}}$ changes. For Karras-style preconditioning,

$$c_{\text{skip}}(t) = \frac{\sigma_{\text{data}}^2}{\sigma_{\text{data}}^2 + t^2},\qquad c_{\text{out}}(t) = \frac{\sigma_{\text{data}}\,t}{\sqrt{\sigma_{\text{data}}^2 + t^2}}.$$

### 7.5 Training: consistency distillation (slides 117–118)

Self-consistency cannot be enforced architecturally — it has to be learned. The training target comes from a **pretrained diffusion model** (the "teacher"):

```
sample x_0 ~ p_data
sample t in [0, T]
let t' = next discretization step before t      (slightly smaller than t)
sample x_t = x_0 + sqrt(t² - 0²) * eps           (noise to time t)

# One step of the PF-ODE backward, using the teacher diffusion model
x_t' = ODESolverStep(x_t, t -> t', teacher)

# Self-consistency loss
L = || f_θ(x_t, t)  -  f_θ_target(x_t', t') ||²
```

Two networks appear in the loss: the **online** network $f_\theta$ (being trained) and the **target** network $f_{\theta^-}$ (a copy with weights updated via EMA: $\theta^- \leftarrow \mu\theta^- + (1-\mu)\theta$). The target gives a stable regression target — same trick as in Q-learning and MoCo.

What the loss is saying: **if you take one ODE step from $x_t$ back to $x_{t'}$, applying $f$ at the new point should give the same answer as applying $f$ at the old point** — because they are on the same trajectory and consistency demands a single answer.

After training, $f_\theta$ has learned to map *any* point on *any* PF-ODE trajectory of the teacher to that trajectory's origin in **one shot**.

### 7.6 Why distillation works

The argument has two pieces:

1. **The chain anchors at $t = 0$.** The boundary condition $f_\theta(x_0, 0) = x_0$ is true by construction at $t = 0$, and self-consistency forces equality at neighbouring $t' > 0$, then at $t > t'$, and so on — like a ladder propagating the correct answer up the trajectory.
2. **The teacher's ODE step is "small enough".** As $\Delta t \to 0$, the teacher's one-step ODE update is asymptotically exact, so the regression target $f_{\theta^-}(x_{t'}, t')$ is a faithful "next rung" on the ladder.

In the limit, a perfectly trained $f_\theta$ is the **exact inverse map** of the teacher's PF-ODE — any point goes back to the origin in one step.

### 7.7 Consistency *training* (without a teacher)

The lecture focuses on **consistency distillation**, which needs a pretrained diffusion model. There is also **consistency training** (CT), which constructs the regression target from independent noise samples instead of an ODE step:

$$x_t = x_0 + t\,\varepsilon,\qquad x_{t'} = x_0 + t'\,\varepsilon\;(\text{same }\varepsilon),$$

so the two perturbed points are on the same noise trajectory by construction. Then minimise $\|f_\theta(x_t, t) - f_{\theta^-}(x_{t'}, t')\|^2$. CT trains a consistency model from scratch with no teacher; later improvements (Improved Consistency Training, Song & Dhariwal 2023) close most of the gap to distillation.

### 7.8 Sampling: trading compute for quality (slide 115)

Once trained, you have a one-step generator: $x_T \sim \mathcal N(0, \sigma_T^2 I)$, $\hat x_0 = f_\theta(x_T, T)$. But sometimes you want *better* samples and have spare compute. Consistency models support a simple **multi-step sampling** procedure:

```
x = sample noise ~ N(0, σ_T² I)
x = f_θ(x, T)               # one-step prediction
for i in 1..N-1:
    z = noise ~ N(0, I)
    x = x + sqrt(t_i² - 0²) * z   # re-noise to a smaller σ
    x = f_θ(x, t_i)               # denoise back
```

Each iteration *re-noises* the prediction to a slightly smaller noise level and then denoises. Because $f_\theta$ is consistent, this preserves the trajectory and refines the sample. Going from 1 → 2 → 4 NFE typically improves sample FID by a noticeable margin.

Same idea also enables **zero-shot image editing**: given an image $x$, partially noise it (to some intermediate $t$), then run consistency sampling — the model fills in the missing parts coherently.

### 7.9 Code: consistency distillation on the 8-Gaussians toy

The next cell visualises the consistency property using the trained DDPM from Part 4. Several PF-ODE trajectories are plotted, and we see how *every* point on a given trajectory should map to its origin under $f_\theta$.


In [ ]:
# ── Consistency model concept visualisation ───────────────────────────────
# Show ODE trajectories and illustrate the consistency property
torch.manual_seed(42)
n_traj = 6
x_init = torch.randn(n_traj, 2) * 2.5

with torch.no_grad():
    paths = simulate_reverse(ddpm_model, x_init, T_diffusion, alpha_bars, betas, stochastic=False, n_steps=100)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(data_2d[:1000, 0], data_2d[:1000, 1], s=1, alpha=0.05, c='gray')
colors = plt.cm.Set1(np.linspace(0, 1, n_traj))
for j in range(n_traj):
    px = [paths[i][j, 0].item() for i in range(len(paths))]
    py = [paths[i][j, 1].item() for i in range(len(paths))]
    ax.plot(px, py, '-', color=colors[j], alpha=0.6, linewidth=2)
    # Mark several points on the trajectory
    for k in [0, len(paths)//4, len(paths)//2, 3*len(paths)//4, -1]:
        ax.plot(paths[k][j, 0].item(), paths[k][j, 1].item(), 'o',
                color=colors[j], markersize=5)
    # Endpoint (x_0)
    ax.plot(paths[-1][j, 0].item(), paths[-1][j, 1].item(), '*',
            color=colors[j], markersize=15, markeredgecolor='black')

ax.set_title('Consistency Models: All points on same trajectory (same colour)\n'
             'map to the SAME endpoint (★)', fontsize=12)
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
plt.tight_layout(); plt.show()
print('A consistency model learns f(x_t, t) = x_0 for ALL points on a trajectory.')
print('Single-step generation: evaluate f(x_T, T) once. No iteration!')

And here is a minimal consistency distillation training loop, distilling our DDPM teacher from Part 4 into a one-step student.


In [ ]:
# ── Consistency distillation (conceptual implementation) ───────────────

class ConsistencyModel(nn.Module):
    """Consistency model with boundary-condition skip connection."""
    def __init__(self, data_dim=2, hidden_dim=128, time_dim=32):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(time_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, data_dim))

    def forward(self, x, t):
        """f(x, t) = c_skip(t) * x + c_out(t) * F(x, t)"""
        # c_skip → 1 as t → 0;  c_out → 0 as t → 0
        t_norm = t.unsqueeze(-1) / T_diffusion  # normalise to [0, 1]
        c_skip = 1.0 / (1.0 + t_norm ** 2)
        c_out = t_norm / (1.0 + t_norm ** 2).sqrt()
        t_emb = self.time_emb(t)
        F_out = self.net(torch.cat([x, t_emb], dim=-1))
        return c_skip * x + c_out * F_out


# Consistency Distillation training sketch (using pre-trained DDPM)
torch.manual_seed(42)
cm = ConsistencyModel()
cm_ema = ConsistencyModel()  # EMA copy
cm_ema.load_state_dict(cm.state_dict())
opt_cm = torch.optim.Adam(cm.parameters(), lr=1e-3)
ema_rate = 0.999

for step in range(3000):
    idx = torch.randperm(len(data_2d))[:256]
    x0 = data_2d[idx]
    # Sample adjacent timesteps t_{n+1} > t_n
    t_next = torch.randint(1, T_diffusion, (256,))
    t_curr = t_next - 1
    # Create noisy x at t_{n+1}
    eps = torch.randn_like(x0)
    ab_next = alpha_bars[t_next].unsqueeze(-1)
    x_next = torch.sqrt(ab_next) * x0 + torch.sqrt(1 - ab_next) * eps
    # One DDPM step: t_{n+1} -> t_n (using pre-trained model)
    with torch.no_grad():
        eps_pred = ddpm_model(x_next, t_next.float())
        ab_curr = alpha_bars[t_curr].unsqueeze(-1)
        a = alphas[t_next].unsqueeze(-1)
        b = betas[t_next].unsqueeze(-1)
        x_curr = (1/torch.sqrt(a)) * (x_next - (b/torch.sqrt(1-ab_next)) * eps_pred)
    # Consistency loss: f(x_{n+1}, t_{n+1}) should equal f_ema(x_n, t_n)
    pred_next = cm(x_next, t_next.float())
    with torch.no_grad():
        pred_curr = cm_ema(x_curr, t_curr.float())
    loss = F.mse_loss(pred_next, pred_curr)
    opt_cm.zero_grad(); loss.backward(); opt_cm.step()
    # EMA update
    with torch.no_grad():
        for p, p_ema in zip(cm.parameters(), cm_ema.parameters()):
            p_ema.data.mul_(ema_rate).add_(p.data, alpha=1 - ema_rate)
    if (step + 1) % 1000 == 0:
        print(f'  Consistency step {step+1}, loss: {loss.item():.4f}')

Finally, the side-by-side comparison: real data vs DDPM samples (300 steps) vs Flow Matching samples (50 steps) vs Consistency model samples (**1 step**).


In [ ]:
# ── Single-step generation with consistency model ───────────────────────
with torch.no_grad():
    z = torch.randn(2000, 2)
    t_max = torch.full((2000,), T_diffusion - 1, dtype=torch.float)
    cm_samples = cm(z * math.sqrt(1 - alpha_bars[-1].item()) + 0, t_max)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ['Real Data', f'DDPM ({T_diffusion} steps)', 'Flow Matching (50 steps)', 'Consistency (1 step!)']
all_samples = [data_2d[:2000], ddpm_samples[:2000], fm_samples[:2000], cm_samples[:2000]]
colors_list = ['tab:blue', 'purple', 'green', 'red']
for ax, title, samp, c in zip(axes, titles, all_samples, colors_list):
    s = samp if isinstance(samp, np.ndarray) else samp.numpy()
    ax.scatter(s[:, 0], s[:, 1], s=1, alpha=0.3, c=c)
    ax.set_title(title); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.suptitle('The Efficiency Trajectory: 300 steps → 50 steps → 1 step', fontsize=14)
plt.tight_layout(); plt.show()

**Read the comparison.** All four should look broadly similar — that's the whole point. The consistency model gets there in **one** network call, the others take 50–300.

### 7.10 Summary of Part 7 (slide 121)

- A **consistency function** maps any point on a PF-ODE trajectory to its origin — by definition, all points on the same trajectory have the same image.
- The **boundary condition** $f(x_0, 0) = x_0$ is enforced architecturally via skip connections.
- **Self-consistency** along trajectories is enforced by training, with a regression target supplied either by a pretrained diffusion teacher (distillation) or by paired noise samples (consistency training).
- The result is a **one-step generative model** that retains diffusion-grade quality and diversity.
- Multi-step sampling trades NFE for quality and supports zero-shot editing.

Consistency models are simultaneously a **distillation technique** for existing diffusion models and a **new generative model family**. They were the state-of-the-art one-step generators on ImageNet at NeurIPS 2023, and they directly inspired the Latent Consistency Models (LCM) and SDXL-Turbo lines that ship in production today.


---
## Part 8 — Grand Comparison and Take-Home Messages

### 8.1 Unified comparison table

| Feature | VAE | GAN | WGAN | DDPM | DDIM | Score / SDE | PF-ODE | Flow Matching | Rectified Flow | Consistency |
|---|---|---|---|---|---|---|---|---|---|---|
| **Training objective** | ELBO | Minimax JS | Minimax $W_1$ | $\varepsilon$-prediction MSE | (same model) | DSM | (same) | Velocity MSE (CFM) | CFM + reflow | Self-consistency |
| **Density** | Lower bound | None | None | ELBO | ELBO | ELBO | Exact via CNF | None / via CNF | None / via CNF | None |
| **Steps to sample** | 1 | 1 | 1 | ~1000 | ~50 | ~1000 | ~10–50 | ~10–50 | ~5 | **1–4** |
| **Training stability** | Stable | Unstable | Stable | Stable | Stable | Stable | Stable | Stable | Stable | Stable (post-DDPM) |
| **Mode coverage** | Excellent | Poor | Better | Excellent | Excellent | Excellent | Excellent | Excellent | Excellent | Excellent |
| **Sample quality** | Blurry | Sharp | Sharp | Excellent | Excellent | Excellent | Excellent | Excellent | Excellent | Excellent |
| **Latent space** | Structured, interpolable | Unstructured | Unstructured | None native | Deterministic via PF-ODE | (same) | (same) | (same) | (same) | (same) |
| **Inversion / editing** | Easy | Hard | Hard | Hard (SDE) | Easy (deterministic) | Hard | Easy | Easy | Easy | Yes (multi-step) |

### 8.2 The efficiency trajectory

```
VAE / GAN:        1 step    (limited quality or stability)
DDPM:             ~1000 steps   (breakthrough quality)
DDIM:             ~50 steps     (same model, faster sampler)
DPM-Solver:       ~10-20 steps  (high-order ODE on the same model)
Flow Matching:    ~10-50 steps  (straighter conditional paths)
Rectified Flow:   ~5 steps      (straighter marginal paths after reflow)
Consistency:      1 step        (PF-ODE inverse, distillation)
```

The story of generative models 2014 → 2023 is one sentence:

> **We sacrificed single-step speed for the quality breakthrough that diffusion brought, then spent five years recovering it without losing the quality.**

### 8.3 Five conceptual unifications you should remember

1. **GAN's optimal discriminator $\Leftrightarrow$ JS divergence.** Vanilla GAN training is JS minimisation when $D$ is optimal, which is also why it fails on disjoint manifolds — JS is constant there.
2. **WGAN's critic $\Leftrightarrow$ Wasserstein-1 in dual form.** The critic estimates $W_1$ via the Kantorovich–Rubinstein duality; weight clipping enforces the Lipschitz constraint.
3. **VAE = ELBO + reparameterisation.** The trick is reparameterisation; without it gradients can't reach the encoder.
4. **DDPM ε-prediction = denoising score matching = $x_0$-prediction = v-prediction.** All four are linear reparameterisations of one underlying object: the score $\nabla_x \log p_t(x)$.
5. **Diffusion = Flow Matching with a Gaussian conditional path.** Flow matching strictly generalises DDPM. Rectified flow generalises further by straightening the marginal field. Consistency models invert the PF-ODE in one step.

### 8.4 What to read next

If you want to go deeper, in roughly increasing order of difficulty:

- **Lilian Weng's blog**, *What Are Diffusion Models?* (2021) — gentle, well-illustrated.
- **Yang Song's blog**, *Generative Modeling by Estimating Gradients of the Data Distribution* (2021) — the score-based view from the inventor.
- **Calvin Luo**, *Understanding Diffusion Models: A Unified Perspective* (arXiv 2208.11970, 2022) — the cleanest derivation of the DDPM ELBO and its connections.
- **Karras, Aittala, Aila, Laine**, *Elucidating the Design Space of Diffusion-Based Generative Models* (NeurIPS 2022) — the EDM paper. Cleans up notation, parameterisation, and training. The de facto modern textbook.
- **Lipman, Chen, Ben-Hamu, Nickel, Le**, *Flow Matching for Generative Modeling* (ICLR 2023).
- **Liu, Gong, Liu**, *Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow* (ICLR 2023).
- **Song, Dhariwal, Chen, Sutskever**, *Consistency Models* (ICML 2023).
- **Esser et al.**, *Scaling Rectified Flow Transformers for High-Resolution Image Synthesis* (ICML 2024) — Stable Diffusion 3.

### 8.5 Self-check: can you answer all of these?

Use these as a rapid self-test. Answers are scattered through the document — if you have to look any up, that's a section worth re-reading.

1. *Why is the marginal log-likelihood $\log p_\theta(x) = \log\int p_\theta(x|z)p(z)dz$ intractable for a VAE? What does the ELBO buy you? What is the gap?*
2. *State the closed-form KL between $\mathcal N(\mu, \mathrm{diag}\,\sigma^2)$ and $\mathcal N(0, I)$.*
3. *What exactly fails in autograd if you replace $z = \mu + \sigma\odot\varepsilon$ with `z = torch.normal(mu, sigma)` inside a VAE forward pass?*
4. *Derive $q(x_t \mid x_0) = \mathcal N(\sqrt{\bar\alpha_t}\,x_0,\,(1-\bar\alpha_t)I)$ from the per-step DDPM forward process.*
5. *Write down (without looking) the simplified DDPM loss and explain why ε-prediction has a friendlier loss landscape than $x_0$-prediction.*
6. *Show that $\nabla_{x_t}\log q(x_t\mid x_0) = -\varepsilon/\sqrt{1-\bar\alpha_t}$ for the DDPM forward kernel. Hence convert a trained $\varepsilon_\theta$ into the marginal score.*
7. *Why does JS divergence "fail" in high dimensions, and what does Wasserstein-1 do differently? What is the role of weight clipping?*
8. *Write the Probability Flow ODE associated with a forward SDE $dx = f(x,t)dt + g(t)dw$. Why does it have the same marginals as the SDE?*
9. *Explain DDIM as Euler's method on the PF-ODE. Why can the same trained model be used at any number of sampling steps?*
10. *State the Conditional Flow Matching objective with the OT path. Why is it equivalent in expectation to the marginal flow matching objective?*
11. *Why are flow-matching trajectories straighter than DDPM trajectories, and how does Rectified Flow make them straighter still?*
12. *Define a consistency function. How is the boundary condition enforced? How is self-consistency enforced?*
13. *Walk through one step of consistency distillation. What is the role of the EMA target network?*
14. *Where does each of {GAN, VAE, DDPM, Flow Matching, Consistency} sit in the (quality, speed, diversity) trilemma?*

If you can answer all 14 from memory, you have understood Week 12.
